# Ranking Fire Detectors by mAP50 Versus Alarm Rate

**A four-detector, three-seed stability audit on D-Fire: complete analysis notebook**

This notebook recomputes every table, figure and in-text number of the paper

> *Ranking Fire Detectors by mAP50 Versus Alarm Rate: A Four-Detector, Three-Seed Stability Audit on D-Fire*, submitted to *Signal, Image and Video Processing* (Springer).

It starts from two kinds of input and nothing else:

1. the curated split manifests of D-Fire (`data/manifests/`, shipped with this repository), which hold the ground-truth boxes and the frozen test partition;
2. the per-image predictions stored by each trained detector down to a confidence of 0.05 (`data/predictions/`, downloaded separately; see the README).

Because the predictions were stored at a confidence far below any operating threshold, every threshold-dependent quantity in the paper (precision, recall, false-alarm rate, fixed-recall comparisons, expected cost) can be recomputed on a CPU without running a network. The only numbers that are *not* recomputed are the GPU measurements in Table 2 (latency, FPS, GFLOPs, peak memory); they are read from the recorded measurement files in `data/recorded/` and displayed as they were measured.

The last section compares everything this notebook computes against the values in the paper and in the original result tables, and prints a PASS/FAIL line for each.

**Runtime.** About one minute on a laptop CPU (0.7 to 1.1 min measured), most of it spent parsing the ResNet18+FPN+FCOS prediction files, each of which holds roughly 700,000 low-confidence boxes. Peak memory is about 1.2 GB. No GPU is needed.

**What the notebook does not do by default.** It does not train anything. An optional appendix shows how the YOLO detectors were trained with Ultralytics under the locked settings, and contains a compact port of the custom ResNet18+FPN+FCOS detector with a CPU smoke test. Both are switched off by flags.

### Where each paper item is produced

| Paper item | Content | Notebook section | Output file |
|---|---|---|---|
| Table 1 | Dataset composition of the curated partitions | 4 | `outputs/tables/Table1_dataset.csv` |
| Table 2 | Parameters, GFLOPs, size, latency, FPS (recorded GPU values) | 11 | `outputs/tables/Table2_capacity.csv` |
| Table 3 | Test results at the validation-selected threshold, mean ± SD over three seeds | 6 | `outputs/tables/Table3_main_results.csv` |
| Table 4 | False alarms per 1,000 verified negatives at fixed recall | 7 | `outputs/tables/Table4_far_at_fixed_recall.csv` |
| Table 5 | Paired bootstrap comparisons, seed-averaged, with the seed-42 column | 8 | `outputs/tables/Table5_paired_comparisons.csv` |
| Table 6 | Smallest attainable two-sided *p* by number of detectors | 9 | `outputs/tables/Table6_permutation_floor.csv` |
| Table 7 | Predictors of the false-alarm ordering | 11 | `outputs/tables/Table7_predictors.csv` |
| Table 8 | Expected cost, rank correlation with mAP50, sweep-floor flag | 10 | `outputs/tables/Table8_expected_cost.csv` |
| Fig. 1 | mAP50 against false-alarm rate, all twelve runs (log axis) | 6 | `outputs/figures/Fig1.png` |
| Fig. 2 | False-alarm rate against accuracy (a) and capacity (b) | 11 | `outputs/figures/Fig2.png` |
| In-text numbers | ρ values, the 81-combination distribution, exact *p*, sweep-floor count, split influence, class gap, multiplicity | 9, 10, 12, 14 | `outputs/tables/intext_numbers.csv` |
| Extended analysis | The 13 tables and 7 figures of the longer version of the paper | 13 | `outputs/tables/ext_*.csv`, `outputs/figures/ExtFig*.png` |
| Verification | Every computed value against the expected value | 14 | `outputs/verification_report.csv` |

All file names, column names and comments are in English. The original analysis code, from which this notebook was ported, used Indonesian identifiers; the logic is unchanged, and Section 14 is the evidence for that claim.

## 1. Configuration

Paths are read from environment variables so that the notebook can be pointed at data stored elsewhere without editing it. Relative defaults are resolved against the working directory, which in Jupyter is the folder that holds this notebook.

| Variable | Default | Meaning |
|---|---|---|
| `FIRE_DATA_DIR` | `data` | Folder with `manifests/`, `recorded/` and, by default, `predictions/` |
| `FIRE_PRED_DIR` | `<FIRE_DATA_DIR>/predictions` | Folder holding one sub-folder per run with `val_dfire.jsonl` and `test_dfire.jsonl`. Several folders may be given, separated by `os.pathsep` (`;` on Windows, `:` elsewhere) |
| `FIRE_OUTPUT_DIR` | `outputs` | Where tables, figures and the verification report are written |

The analysis settings below were fixed before any detector was trained and are not tuned here. They reproduce, value for value, the two configuration files of the original analysis (`config/pengaturan.yaml` for measurement and statistics, `config/biaya.yaml` for the cost ratios) and the locked evaluation settings of the training pipeline (`configs/training_defaults.yaml`: predictions kept down to confidence 0.05, NMS IoU 0.60, 2,000 bootstrap resamples).

In [ ]:
from __future__ import annotations

import hashlib
import io
import itertools
import json
import math
import os
import platform
import sys
import time
from dataclasses import dataclass
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path
from typing import Any

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_START = time.time()
# Tables below print symbols such as tau and plus-minus. A plain Windows console may not encode
# them; this keeps printing safe when the code runs outside Jupyter.
if hasattr(sys.stdout, "reconfigure") and (sys.stdout.encoding or "").lower().replace("-", "") != "utf8":
    sys.stdout.reconfigure(errors="replace")
print(
    f"Python {platform.python_version()} | numpy {np.__version__} | "
    f"pandas {pd.__version__} | matplotlib {matplotlib.__version__}"
)


try:
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821  (inside Jupyter only)
except NameError:
    pass   # plain script: matplotlib keeps its default (or MPLBACKEND) backend


def show(obj: Any, title: str | None = None) -> None:
    """Display a table in Jupyter, or print it when the code runs as a plain script."""
    if title:
        print(f"\n{title}")
    try:
        get_ipython  # noqa: B018  (only defined inside IPython)
        from IPython.display import display

        display(obj)
    except NameError:
        print(obj.to_string(index=False) if isinstance(obj, pd.DataFrame) else obj)

In [ ]:
# --- Paths ---------------------------------------------------------------------------------
DATA_DIR = Path(os.environ.get("FIRE_DATA_DIR", "data")).expanduser().resolve()
_pred_env = os.environ.get("FIRE_PRED_DIR", "").strip()
PRED_DIRS = [Path(p).expanduser().resolve() for p in _pred_env.split(os.pathsep) if p.strip()]
if not PRED_DIRS:
    PRED_DIRS = [DATA_DIR / "predictions"]
OUTPUT_DIR = Path(os.environ.get("FIRE_OUTPUT_DIR", "outputs")).expanduser().resolve()

MANIFEST_DIR = DATA_DIR / "manifests"
RECORDED_DIR = DATA_DIR / "recorded"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not MANIFEST_DIR.is_dir():
    raise FileNotFoundError(
        f"The manifest folder was not found at {MANIFEST_DIR}.\n"
        "Run the notebook from the repository root (the folder that contains data/), or set "
        "FIRE_DATA_DIR to the folder that contains manifests/ and recorded/."
    )

print(f"data folder        : {DATA_DIR}")
print(f"prediction folders : {', '.join(str(p) for p in PRED_DIRS)}")
print(f"output folder      : {OUTPUT_DIR}")

In [ ]:
# --- Locked analysis settings ---------------------------------------------------------------
# These values were fixed in the configuration of the original analysis and of the training
# pipeline before any detector was trained (see the Markdown above).

CLASSES: tuple[str, ...] = ("fire", "smoke")      # class index 0 = fire, 1 = smoke
SEEDS: tuple[int, ...] = (42, 1337, 2024)          # training seeds of every detector

IOU_MAIN = 0.5                                     # AP50, point metrics and matching
IOU_THRESHOLDS = np.round(np.arange(0.50, 1.0, 0.05), 2)   # mAP50-95, COCO convention
RECALL_POINTS = np.linspace(0.0, 1.0, 101)         # 101-point interpolated precision

# Confidence thresholds swept on validation: 0.05 to 0.95 in steps of 0.05.
THRESHOLD_SWEEP = np.round(np.arange(0.05, 0.95 + 1e-9, 0.05), 2)
STORED_CONFIDENCE_FLOOR = 0.05                     # lowest confidence kept at prediction time

FIXED_RECALLS: tuple[float, ...] = (0.5, 0.6, 0.7, 0.8)      # Table 4
COST_RATIOS: tuple[int, ...] = (1, 2, 5, 10, 25, 50)         # C_miss / C_false-alarm, Table 8

BOOTSTRAP_RESAMPLES = 2000
CONFIDENCE_LEVEL = 0.95
RANDOM_SEED = 42                                   # seed of every bootstrap generator

# Integrity of the frozen test partition: content hash and image counts.
FROZEN_TEST_SHA256 = "6246e1bcca7947392097d8460a33e4e5e5823c8ff3c20ebc6d852d3c1af56ae9"
EXPECTED_IMAGES = {"train": 14852, "val": 3418, "test": 3257}

# Official D-Fire partition used by the split-influence control (built from the native
# folder of every image; see Section 2.3).
EXPECTED_OFFICIAL_IMAGES = {"train": 14496, "val": 2725, "test": 4306}

# Reproduction check: values verified in the training pipeline for the custom detector at seed 42,
# with the tolerance the original check applied to each of them.
REFERENCE_RUN = "resnet18fpn_seed42"
REFERENCE_VALUES = {
    "map50": (0.66107, 0.0005),
    "ap_fire": (0.57554, 0.0005),
    "ap_smoke": (0.74659, 0.0005),
    "far_per_1000": (11.863, 0.05),
    "threshold": (0.45, 0.001),
}

# Internal consistency of the settings above.
assert len(IOU_THRESHOLDS) == 10 and IOU_THRESHOLDS[0] == 0.5 and IOU_THRESHOLDS[-1] == 0.95
assert len(THRESHOLD_SWEEP) == 19 and THRESHOLD_SWEEP[0] == 0.05 and THRESHOLD_SWEEP[-1] == 0.95
assert np.array_equal(THRESHOLD_SWEEP, np.round(np.arange(0.05, 0.96, 0.05), 2))
assert THRESHOLD_SWEEP[0] == STORED_CONFIDENCE_FLOOR

show(pd.DataFrame([
    {"setting": "classes", "value": ", ".join(CLASSES)},
    {"setting": "training seeds", "value": ", ".join(map(str, SEEDS))},
    {"setting": "IoU for AP50, P/R/F1 and matching", "value": IOU_MAIN},
    {"setting": "IoU thresholds for mAP50-95", "value": f"{IOU_THRESHOLDS[0]:.2f} to {IOU_THRESHOLDS[-1]:.2f}, step 0.05"},
    {"setting": "confidence sweep (selection on validation)", "value": "0.05 to 0.95, step 0.05"},
    {"setting": "stored confidence floor", "value": STORED_CONFIDENCE_FLOOR},
    {"setting": "fixed recalls", "value": ", ".join(map(str, FIXED_RECALLS))},
    {"setting": "cost ratios C_miss / C_false-alarm", "value": ", ".join(map(str, COST_RATIOS))},
    {"setting": "bootstrap resamples / level / seed", "value": f"{BOOTSTRAP_RESAMPLES} / {CONFIDENCE_LEVEL} / {RANDOM_SEED}"},
    {"setting": "frozen test manifest SHA-256", "value": FROZEN_TEST_SHA256},
]), "Locked settings")

### 1.1 Runs

Fourteen training runs enter the analysis: four detectors with three seeds each on the curated (deduplicated) partition, and YOLO11n retrained with two seeds on the official D-Fire partition. Each run has an English identifier used throughout this notebook and the folder name under which its predictions were stored by the original project. The experiment codes (E0, E0b, E9, E10, E11) are those of the training pipeline and appear in the `_meta` line of every prediction file.

In [ ]:
DETECTORS: tuple[str, ...] = ("ResNet18+FPN+FCOS", "YOLO11n", "YOLO11s", "YOLOv8s")
SHORT_NAME = {"ResNet18+FPN+FCOS": "ResNet", "YOLO11n": "YOLO11n", "YOLO11s": "YOLO11s",
              "YOLOv8s": "YOLOv8s"}
FAMILY = {"ResNet18+FPN+FCOS": "custom single-stage", "YOLO11n": "YOLO", "YOLO11s": "YOLO",
          "YOLOv8s": "YOLO"}
EXPERIMENT_CODE = {"ResNet18+FPN+FCOS": "E0b", "YOLO11n": "E0", "YOLO11s": "E9", "YOLOv8s": "E10"}
_FOLDER_PREFIX = {"ResNet18+FPN+FCOS": "resnet18fpn", "YOLO11n": "yolo11n", "YOLO11s": "yolo11s",
                  "YOLOv8s": "yolov8s"}


@dataclass(frozen=True)
class Run:
    run_id: str          # English identifier used in this notebook
    stored_folder: str   # folder name under which the predictions were stored
    detector: str
    seed: int
    partition: str       # "curated" (deduplicated, frozen test) or "official" (D-Fire folders)
    experiment: str


RUNS: dict[str, Run] = {}
for _det in DETECTORS:
    for _seed in SEEDS:
        _rid = f"{_FOLDER_PREFIX[_det]}_seed{_seed}"
        RUNS[_rid] = Run(_rid, _rid, _det, _seed, "curated", EXPERIMENT_CODE[_det])
for _seed in (42, 1337):
    RUNS[f"yolo11n_official_seed{_seed}"] = Run(
        f"yolo11n_official_seed{_seed}", f"yolo11n_split_resmi_seed{_seed}", "YOLO11n", _seed,
        "official", "E11",
    )

CURATED_RUNS = [r for r in RUNS.values() if r.partition == "curated"]
OFFICIAL_RUNS = [r for r in RUNS.values() if r.partition == "official"]
show(pd.DataFrame([r.__dict__ for r in RUNS.values()]), "Runs analysed")

## 2. Data loading and integrity checks

### 2.1 Manifests and their content hashes

A manifest lists the images of one partition together with their boxes, stored as normalised centre/size coordinates in JSON strings. The training pipeline registered an order-independent SHA-256 of each manifest's content in `MANIFEST_HASHES.json`: every row is serialised as canonical JSON and hashed, the row hashes are sorted, and the sorted list is hashed again. Recomputing that hash here, rather than trusting the registry, is what shows that the ground truth used below is the one the detectors were evaluated against. The test manifest was frozen before any detector in the study was trained; if its hash differs, the analysis stops.

In [ ]:
def _builtin(value: Any) -> Any:
    """Convert numpy scalars to Python scalars so that JSON serialisation is canonical."""
    if isinstance(value, np.generic):
        return value.item()
    return value


def _sha256_json(obj: Any) -> str:
    payload = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def manifest_content_sha256(frame: pd.DataFrame) -> str:
    """Order-independent content hash, identical to the one registered by the training pipeline."""
    rows = frame.astype(object).where(frame.notna(), None).to_dict(orient="records")
    row_hashes = sorted(_sha256_json({k: _builtin(v) for k, v in row.items()}) for row in rows)
    return _sha256_json(row_hashes)


def load_manifest(split: str) -> pd.DataFrame:
    path = MANIFEST_DIR / f"dfire_{split}.parquet"
    if not path.exists():
        raise FileNotFoundError(
            f"Manifest {path.name} is missing from {MANIFEST_DIR}. It is part of this "
            "repository; restore it from version control."
        )
    return pd.read_parquet(path)


registry = json.loads((MANIFEST_DIR / "MANIFEST_HASHES.json").read_text(encoding="utf-8"))["manifests"]
MANIFESTS: dict[str, pd.DataFrame] = {}
integrity_rows = []
for _split in ("train", "val", "test"):
    frame = load_manifest(_split)
    entry = registry[f"dfire_{_split}"]
    computed = manifest_content_sha256(frame)
    integrity_rows.append({
        "partition": _split,
        "images": len(frame),
        "expected_images": EXPECTED_IMAGES[_split],
        "content_sha256": computed,
        "matches_registry": computed == entry["content_sha256"],
        "frozen": bool(entry.get("immutable", False)),
    })
    if len(frame) != EXPECTED_IMAGES[_split]:
        raise RuntimeError(f"{_split}: {len(frame)} images, expected {EXPECTED_IMAGES[_split]}.")
    if computed != entry["content_sha256"]:
        raise RuntimeError(
            f"{_split}: the manifest content hash {computed[:12]} does not match the registered "
            f"{entry['content_sha256'][:12]}. The file was modified; no result computed from it is valid."
        )
    MANIFESTS[_split] = frame

if integrity_rows[2]["content_sha256"] != FROZEN_TEST_SHA256 or not integrity_rows[2]["frozen"]:
    raise RuntimeError("The frozen test manifest does not carry the locked hash. Stopping.")
INTEGRITY = pd.DataFrame(integrity_rows)
show(INTEGRITY, "Manifest integrity (all three hashes recomputed from content)")
print(f"Frozen test manifest verified: {FROZEN_TEST_SHA256}")

### 2.2 Ground truth

Boxes are converted from normalised centre/size to pixel corners using each image's original size, and clipped to the image. An image whose annotation file exists and is empty is a **verified negative**: annotators looked at it and found no fire or smoke. Those images are the denominator of the false-alarm rate. D-Fire annotates every image, so no image is excluded as "unknown"; the distinction matters only because an unannotated image must never be counted as a negative.

In [ ]:
@dataclass
class GroundTruth:
    name: str
    sample_ids: list[str]
    boxes: dict[str, list[np.ndarray]]   # class -> one (G, 4) float32 array per image, xyxy pixels
    is_negative: np.ndarray              # True where the image has no box at all

    @property
    def negative_ids(self) -> list[str]:
        return [sid for sid, neg in zip(self.sample_ids, self.is_negative) if neg]

    @property
    def n_negatives(self) -> int:
        return int(self.is_negative.sum())


def _parse_boxes(cell: Any) -> list[dict[str, Any]]:
    if cell is None:
        return []
    return json.loads(cell) if isinstance(cell, str) else list(cell)


def build_ground_truth(frame: pd.DataFrame, name: str) -> GroundTruth:
    """Pixel-space boxes per class and per image, in manifest order.

    Manifest order is kept on purpose: it fixes the order in which predictions enter the
    precision-recall computation and the order of negatives in the bootstrap, both of which
    must match the original analysis for the results to agree to the last digit.
    """
    ids: list[str] = []
    per_class: dict[str, list[np.ndarray]] = {c: [] for c in CLASSES}
    negative: list[bool] = []
    for sid, width, height, fire, smoke in zip(
        frame.sample_id, frame.width, frame.height, frame.fire_boxes, frame.smoke_boxes
    ):
        w, h = int(width or 1), int(height or 1)
        found: dict[str, list[list[float]]] = {c: [] for c in CLASSES}
        n_boxes = 0
        for cell in (fire, smoke):
            for b in _parse_boxes(cell):
                x1 = (b["cx"] - b["w"] / 2) * w
                y1 = (b["cy"] - b["h"] / 2) * h
                x2 = (b["cx"] + b["w"] / 2) * w
                y2 = (b["cy"] + b["h"] / 2) * h
                n_boxes += 1
                if b["cls"] in found:
                    found[b["cls"]].append([max(0.0, x1), max(0.0, y1), min(float(w), x2), min(float(h), y2)])
        ids.append(sid)
        for c in CLASSES:
            per_class[c].append(np.asarray(found[c], dtype=np.float32).reshape(-1, 4))
        negative.append(n_boxes == 0)
    return GroundTruth(name, ids, per_class, np.asarray(negative, dtype=bool))


GT: dict[str, GroundTruth] = {s: build_ground_truth(MANIFESTS[s], s) for s in ("val", "test")}
for _s, _g in GT.items():
    print(f"{_s:5s}: {len(_g.sample_ids):5d} images, {_g.n_negatives:5d} verified negatives, "
          + ", ".join(f"{sum(len(b) for b in _g.boxes[c])} {c} boxes" for c in CLASSES))

### 2.3 The official D-Fire partition

The split-influence control (Section 12) retrains YOLO11n on D-Fire's own folders instead of our deduplicated partition. Its partition is not a separate download: every manifest row records the release folder the image came from (`metadata.native_split`), so the official partition can be rebuilt from the three curated manifests exactly as the training pipeline built it:

* **test**: every image from the release's `test` folder (4,306 images), in the order train, val, test of the curated manifests;
* **validation**: the curated validation images that came from the release's `train` folder (2,725), so no new random draw is involved;
* **train**: the remaining images of the release's `train` folder (14,496).

The same bookkeeping shows why the control needs a retrained model at all: 2,990 images of the official test folder sit inside our curated training partition, so evaluating our curated models on the official test folder would be leakage.

In [ ]:
def native_folder(metadata: Any) -> str | None:
    meta = json.loads(metadata) if isinstance(metadata, str) else (metadata or {})
    return (meta or {}).get("native_split")


_all_rows = pd.concat([MANIFESTS["train"], MANIFESTS["val"], MANIFESTS["test"]], ignore_index=True)
_native = _all_rows.metadata.map(native_folder)
_val_native = MANIFESTS["val"].metadata.map(native_folder)

OFFICIAL_MANIFESTS = {
    "val": MANIFESTS["val"][(_val_native == "train").to_numpy()].reset_index(drop=True),
    "test": _all_rows[(_native == "test").to_numpy()].reset_index(drop=True),
}
_official_val_ids = set(OFFICIAL_MANIFESTS["val"].sample_id)
OFFICIAL_MANIFESTS["train"] = _all_rows[
    ((_native == "train") & ~_all_rows.sample_id.isin(_official_val_ids)).to_numpy()
].reset_index(drop=True)

OFFICIAL_COUNTS = {k: len(v) for k, v in OFFICIAL_MANIFESTS.items()}
LEAKED_OFFICIAL_TEST_IN_CURATED_TRAIN = int(
    (MANIFESTS["train"].metadata.map(native_folder) == "test").sum()
)
_overlap = (set(OFFICIAL_MANIFESTS["train"].sample_id) | _official_val_ids) & set(
    OFFICIAL_MANIFESTS["test"].sample_id
)
for _k, _n in EXPECTED_OFFICIAL_IMAGES.items():
    if OFFICIAL_COUNTS[_k] != _n:
        raise RuntimeError(f"official {_k}: {OFFICIAL_COUNTS[_k]} images, expected {_n}")
if _overlap:
    raise RuntimeError(f"{len(_overlap)} official test images also appear in official train/val")

GT_OFFICIAL = {s: build_ground_truth(OFFICIAL_MANIFESTS[s], f"official_{s}") for s in ("val", "test")}
print(f"official partition: train {OFFICIAL_COUNTS['train']}, val {OFFICIAL_COUNTS['val']}, "
      f"test {OFFICIAL_COUNTS['test']} ({GT_OFFICIAL['test'].n_negatives} verified negatives); "
      f"no overlap between official train/val and test")
print(f"official test images inside the curated training partition: "
      f"{LEAKED_OFFICIAL_TEST_IN_CURATED_TRAIN}")

### 2.4 Stored predictions

Each run has two files, `val_dfire.jsonl` and `test_dfire.jsonl`. The first line may be a `_meta` record (partition, experiment code, the confidence floor used at prediction time); every other line holds one image:

```json
{"sample_id": "dfire/test/images/AoF06735", "source": "dfire",
 "detections": [{"bbox": [88.97, 307.84, 117.31, 355.93], "score": 0.19945, "class_idx": 1, "class": "smoke"}]}
```

Boxes are corner coordinates in original-image pixels after non-maximum suppression at IoU 0.60; scores are rounded to five decimals. The loader keeps every detection of a file in three flat arrays, in file order, and records the row range belonging to each image. Scores are kept in double precision exactly as parsed; the original analysis compared them in single precision in some places and in double precision in others, and Section 3 makes each of those choices explicit.

The loader looks for each run in every folder of `FIRE_PRED_DIR`, under its stored folder name or its English run identifier. It also understands the layout of the original analysis project (`apps/models/<run>/prediksi/` and `apps/models_versi2/<run>/prediksi/`), so it can be pointed at that project directly.

In [ ]:
CLASS_INDEX = {c: i for i, c in enumerate(CLASSES)}


@dataclass
class Predictions:
    run_id: str
    split: str
    rows: dict[str, slice]       # sample_id -> slice into the flat arrays
    boxes: np.ndarray            # (M, 4) float32, xyxy pixels
    scores: np.ndarray           # (M,) float64, as stored in the file
    class_idx: np.ndarray        # (M,) int8, index into CLASSES; -1 for any other label
    meta: dict[str, Any]

    @property
    def n_detections(self) -> int:
        return int(len(self.scores))


def prediction_candidates(run: Run, split: str) -> list[Path]:
    name = f"{split}_dfire.jsonl"
    out: list[Path] = []
    for root in PRED_DIRS:
        for folder in dict.fromkeys((run.stored_folder, run.run_id)):
            out += [root / folder / name, root / folder / "prediksi" / name]
        # Original project layout: the second batch of runs lives in a sibling folder.
        out.append(root.parent / "models_versi2" / run.stored_folder / "prediksi" / name)
    return out


def find_prediction_file(run: Run, split: str) -> Path | None:
    return next((p for p in prediction_candidates(run, split) if p.is_file()), None)


def load_predictions(path: Path, run_id: str, split: str) -> Predictions:
    rows: dict[str, slice] = {}
    boxes: list[list[float]] = []
    scores: list[float] = []
    classes: list[int] = []
    meta: dict[str, Any] = {}
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if "_meta" in record:
                meta = record["_meta"]
                continue
            start = len(scores)
            for det in record.get("detections", []):
                boxes.append(det["bbox"])
                scores.append(float(det["score"]))
                classes.append(CLASS_INDEX.get(det["class"], -1))
            rows[record["sample_id"]] = slice(start, len(scores))
    return Predictions(
        run_id, split, rows,
        np.asarray(boxes, dtype=np.float64).reshape(-1, 4).astype(np.float32),
        np.asarray(scores, dtype=np.float64),
        np.asarray(classes, dtype=np.int8),
        meta,
    )


_missing = [(r, s) for r in RUNS.values() for s in ("val", "test") if find_prediction_file(r, s) is None]
if _missing:
    raise FileNotFoundError(
        "Stored predictions are missing for "
        + ", ".join(f"{r.stored_folder}/{s}_dfire.jsonl" for r, s in _missing)
        + ".\n\nDownload the prediction archive described in the README and place one folder per run "
        f"under {PRED_DIRS[0]}, for example\n  {PRED_DIRS[0] / 'resnet18fpn_seed42' / 'test_dfire.jsonl'}\n"
        "or set FIRE_PRED_DIR to the folder that contains the run folders."
    )

PREDICTIONS: dict[tuple[str, str], Predictions] = {}
_load_rows = []
for _run in RUNS.values():
    for _split in ("val", "test"):
        _t0 = time.time()
        _path = find_prediction_file(_run, _split)
        _pred = load_predictions(_path, _run.run_id, _split)
        _gt = (GT_OFFICIAL if _run.partition == "official" else GT)[_split]
        _covered = sum(sid in _pred.rows for sid in _gt.sample_ids)
        if _covered != len(_gt.sample_ids):
            raise RuntimeError(
                f"{_path} covers {_covered} of the {len(_gt.sample_ids)} images of its partition; "
                "the file is incomplete or belongs to a different partition."
            )
        floor = _pred.meta.get("conf_threshold_used", STORED_CONFIDENCE_FLOOR)
        if floor > STORED_CONFIDENCE_FLOOR + 1e-12 or (_pred.n_detections and _pred.scores.min() < floor - 1e-9):
            raise RuntimeError(f"{_path}: predictions were not stored down to {STORED_CONFIDENCE_FLOOR}.")
        PREDICTIONS[(_run.run_id, _split)] = _pred
        _load_rows.append({"run": _run.run_id, "split": _split, "images": len(_pred.rows),
                           "detections": _pred.n_detections,
                           "experiment": _pred.meta.get("experiment_id", ""),
                           "seconds": round(time.time() - _t0, 1)})
        if _pred.meta.get("experiment_id") not in (None, _run.experiment):
            raise RuntimeError(f"{_path} belongs to experiment {_pred.meta.get('experiment_id')}, "
                               f"expected {_run.experiment}.")
show(pd.DataFrame(_load_rows), "Prediction files loaded")

## 3. Metrics and the reproduction check

### 3.1 Detection metrics

The conventions follow pycocotools and the training pipeline that scored every detector:

* IoU between a detection and a ground-truth box of the same class;
* greedy matching: within an image, detections are visited from the highest score down, and each takes the unmatched ground-truth box with the highest IoU, provided that IoU reaches the threshold;
* AP with 101-point interpolated precision. The recall lookup uses `searchsorted(..., side="left")`, not linear interpolation; with interpolation a perfect detector would score 100/101 instead of 1;
* mAP50 is the mean AP over fire and smoke at IoU 0.50; mAP50-95 averages ten IoU thresholds;
* precision, recall and F1 at a confidence threshold τ count only detections with score ≥ τ;
* the false-alarm rate counts, per 1,000 verified negatives, the images with at least one detection of any class at score ≥ τ. An image counts once however many boxes it carries.

**Numerical detail that matters for exact agreement.** The original analysis held scores in single precision (float32) for matching and AP, compared them with thresholds in single precision when selecting thresholds and computing expected cost, in double precision on the parsed scores for the false-alarm rate, and in double precision on the float32 scores for the threshold curves of Section 7. A score stored as `0.45000` is 0.44999998 in float32, so these choices decide whether a detection sitting exactly on a threshold is counted. Each comparison below states its precision explicitly, so the result does not depend on the numpy version's promotion rules. On the stored predictions these choices move a single point of one threshold curve (YOLO11s, seed 42, τ = 0.65) and none of the reported numbers, but keeping them makes the tables and figures reproduce byte for byte.

In [ ]:
def iou_matrix(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    """IoU of every prediction against every ground-truth box: (N, 4), (M, 4) -> (N, M) float32."""
    if len(pred) == 0 or len(gt) == 0:
        return np.zeros((len(pred), len(gt)), dtype=np.float32)
    p = pred[:, None, :]
    g = gt[None, :, :]
    x1 = np.maximum(p[..., 0], g[..., 0])
    y1 = np.maximum(p[..., 1], g[..., 1])
    x2 = np.minimum(p[..., 2], g[..., 2])
    y2 = np.minimum(p[..., 3], g[..., 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_p = np.clip(p[..., 2] - p[..., 0], 0, None) * np.clip(p[..., 3] - p[..., 1], 0, None)
    area_g = np.clip(g[..., 2] - g[..., 0], 0, None) * np.clip(g[..., 3] - g[..., 1], 0, None)
    return (inter / np.clip(area_p + area_g - inter, 1e-9, None)).astype(np.float32)


def average_precision(tp: np.ndarray, scores: np.ndarray, n_gt: int) -> float:
    """101-point interpolated AP from per-detection true-positive flags (float32 arithmetic)."""
    if n_gt == 0:
        return float("nan")
    order = np.argsort(-scores, kind="stable")
    tp = tp[order].astype(np.float32)
    tp_cum = np.cumsum(tp, dtype=np.float32)
    fp_cum = np.cumsum(np.float32(1) - tp, dtype=np.float32)
    recall = tp_cum / np.float32(n_gt)
    precision = tp_cum / np.clip(tp_cum + fp_cum, np.float32(1e-9), None)
    # Monotone envelope: a local dip in precision must not lower AP.
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    interpolated = np.zeros_like(RECALL_POINTS)
    idx = np.searchsorted(recall, RECALL_POINTS, side="left")
    valid = idx < len(precision)
    interpolated[valid] = precision[idx[valid]]
    return float(interpolated.mean())


@dataclass
class ClassMatch:
    scores: np.ndarray   # float32 scores in evaluation order (image order, then file order)
    tp: np.ndarray       # float32 true-positive flags at IoU 0.50, aligned with scores
    n_gt: int


def _class_items(preds: Predictions, gt: GroundTruth, c_idx: int) -> list[tuple[int, np.ndarray, np.ndarray]]:
    """(image position, boxes, float32 scores) for every image with detections of one class."""
    items = []
    for i, sid in enumerate(gt.sample_ids):
        sl = preds.rows.get(sid)
        if sl is None or sl.start == sl.stop:
            continue
        mask = preds.class_idx[sl] == c_idx
        if mask.any():
            items.append((i, preds.boxes[sl][mask], preds.scores[sl][mask].astype(np.float32)))
    return items


def _greedy_tp(n: int, iou: np.ndarray, iou64: np.ndarray, best: np.ndarray,
               order: np.ndarray, threshold: float) -> np.ndarray:
    """Greedy score-ordered matching within one image at one IoU threshold.

    A detection whose best IoU is already below the threshold cannot match anything; skipping it
    gives the same flags as visiting it and stopping at the first candidate, only faster. IoU is
    compared in double precision against the double-precision threshold, as in the original code.
    """
    tp = np.zeros(n, dtype=np.float32)
    used: set[int] = set()
    for k in order:
        if best[k] < threshold:
            continue
        for g in np.argsort(-iou[k]):
            if iou64[k, g] < threshold:
                break
            if int(g) not in used:
                used.add(int(g))
                tp[k] = 1.0
                break
    return tp


def match_class(items: list[tuple[int, np.ndarray, np.ndarray]], gt_boxes: list[np.ndarray],
                iou_thresholds: Any) -> list[np.ndarray]:
    """True-positive flags for one class at each IoU threshold, concatenated in evaluation order."""
    prepared = []
    for i, boxes, scores in items:
        g = gt_boxes[i]
        if len(g) == 0:
            prepared.append((len(scores), None))
            continue
        iou = iou_matrix(boxes, g)
        iou64 = iou.astype(np.float64)
        prepared.append((len(scores), (iou, iou64, iou64.max(axis=1), np.argsort(-scores, kind="stable"))))
    flags = []
    for thr in iou_thresholds:
        parts = [np.zeros(n, np.float32) if prep is None else _greedy_tp(n, *prep, float(thr))
                 for n, prep in prepared]
        flags.append(np.concatenate(parts) if parts else np.zeros(0, np.float32))
    return flags


def match_at_iou(preds: Predictions, gt: GroundTruth, iou_threshold: float = IOU_MAIN) -> dict[str, ClassMatch]:
    """Match once at a single IoU; thresholds on confidence then only filter by score."""
    out = {}
    for c_idx, c in enumerate(CLASSES):
        n_gt = int(sum(len(b) for b in gt.boxes[c]))
        items = _class_items(preds, gt, c_idx)
        scores = np.concatenate([s for _, _, s in items]) if items else np.zeros(0, np.float32)
        tp = match_class(items, gt.boxes[c], [iou_threshold])[0] if items else np.zeros(0, np.float32)
        out[c] = ClassMatch(scores, tp, n_gt)
    return out


def negative_max_scores(preds: Predictions, gt: GroundTruth) -> tuple[np.ndarray, np.ndarray]:
    """Highest score per verified negative: per class (float32) and over all labels (float64).

    An image raises an alarm at threshold t exactly when its highest score reaches t, so these
    two arrays are enough to compute the false-alarm rate at any threshold without the files.
    """
    neg_ids = gt.negative_ids
    by_class = np.zeros((len(neg_ids), len(CLASSES)), dtype=np.float32)
    any_label = np.full(len(neg_ids), -np.inf)
    for i, sid in enumerate(neg_ids):
        sl = preds.rows.get(sid)
        if sl is None or sl.start == sl.stop:
            continue
        s, ci = preds.scores[sl], preds.class_idx[sl]
        any_label[i] = s.max()
        for j in range(len(CLASSES)):
            mask = ci == j
            if mask.any():
                by_class[i, j] = s[mask].max()
    return by_class, any_label


def false_alarm_rate(neg_any_label: np.ndarray, threshold: float) -> tuple[float | None, int, int]:
    """(alarms per 1,000 verified negatives, negatives, negatives that alarm); double precision."""
    n_neg = len(neg_any_label)
    if n_neg == 0:
        return None, 0, 0
    alarmed = int((neg_any_label >= float(threshold)).sum())
    return round(1000.0 * alarmed / n_neg, 3), n_neg, alarmed


def _mean_defined(values: dict[str, float | None]) -> float | None:
    defined = [v for v in values.values() if v is not None]
    return round(float(np.mean(defined)), 5) if defined else None


def evaluate_detection(preds: Predictions, gt: GroundTruth, threshold: float) -> tuple[dict[str, Any], dict[str, ClassMatch]]:
    """Full evaluation of one run on one partition at one confidence threshold."""
    res: dict[str, Any] = {k: {} for k in ("ap50", "ap50_95", "precision", "recall", "f1", "n_gt", "n_pred")}
    matches: dict[str, ClassMatch] = {}
    thr32 = np.float32(threshold)   # point metrics compare float32 scores in float32
    for c_idx, c in enumerate(CLASSES):
        n_gt = int(sum(len(b) for b in gt.boxes[c]))
        items = _class_items(preds, gt, c_idx)
        res["n_gt"][c] = n_gt
        res["n_pred"][c] = int(sum(len(s) for _, _, s in items))
        if n_gt == 0:
            for k in ("ap50", "ap50_95", "precision", "recall", "f1"):
                res[k][c] = None     # undefined, never zero
            matches[c] = ClassMatch(np.zeros(0, np.float32), np.zeros(0, np.float32), 0)
            continue
        if not items:
            for k in ("ap50", "ap50_95", "precision", "recall", "f1"):
                res[k][c] = 0.0
            matches[c] = ClassMatch(np.zeros(0, np.float32), np.zeros(0, np.float32), n_gt)
            continue
        scores = np.concatenate([s for _, _, s in items])
        flags = match_class(items, gt.boxes[c], IOU_THRESHOLDS)
        aps = [average_precision(tp, scores, n_gt) for tp in flags]
        res["ap50"][c] = round(aps[0], 5)
        selected = scores >= thr32
        tp_n = float(flags[0][selected].sum())
        fp_n = float(selected.sum() - tp_n)
        precision = tp_n / max(1e-9, tp_n + fp_n)
        recall = tp_n / max(1e-9, n_gt)
        res["precision"][c] = round(precision, 5)
        res["recall"][c] = round(recall, 5)
        res["f1"][c] = round(2 * precision * recall / max(1e-9, precision + recall), 5)
        defined = [a for a in aps if not np.isnan(a)]
        res["ap50_95"][c] = round(float(np.mean(defined)), 5) if defined else None
        matches[c] = ClassMatch(scores, flags[0], n_gt)

    res["map50"] = _mean_defined(res["ap50"])
    res["map50_95"] = _mean_defined(res["ap50_95"])
    res["mean_precision"] = _mean_defined(res["precision"])
    res["mean_recall"] = _mean_defined(res["recall"])
    res["mean_f1"] = _mean_defined(res["f1"])
    _, neg_any = negative_max_scores(preds, gt)
    res["far_per_1000"], res["n_negatives"], res["n_negatives_alarmed"] = false_alarm_rate(neg_any, threshold)
    res["threshold"] = float(threshold)
    return res, matches

### 3.2 Operating point: the threshold is chosen on validation only

For each run the confidence threshold τ is swept from 0.05 to 0.95 on the **validation** partition, and the value with the highest mean F1 over the two classes is carried unchanged to the test partition. Per-class F1 is rounded to five decimals before averaging and ties go to the lower threshold, which is how the training pipeline selected the thresholds the paper reports. Choosing τ on the test partition would tune a hyperparameter on the data used to report the result.

In [ ]:
def select_threshold(val_matches: dict[str, ClassMatch]) -> tuple[float, pd.DataFrame]:
    """Validation sweep; returns the selected threshold and the full sweep."""
    rows = []
    for t in THRESHOLD_SWEEP:
        t32 = np.float32(t)
        p_c, r_c, f_c = [], [], []
        for c in CLASSES:
            m = val_matches[c]
            if m.n_gt == 0:
                continue
            selected = m.scores >= t32
            tp_n = float(m.tp[selected].sum())
            fp_n = float(selected.sum() - tp_n)
            p = tp_n / max(1e-9, tp_n + fp_n)
            r = tp_n / max(1e-9, m.n_gt)
            p_c.append(round(p, 5))
            r_c.append(round(r, 5))
            f_c.append(round(2 * p * r / max(1e-9, p + r), 5))
        rows.append({"threshold": float(t), "mean_f1": round(float(np.mean(f_c)), 5),
                     "mean_precision": round(float(np.mean(p_c)), 5),
                     "mean_recall": round(float(np.mean(r_c)), 5)})
    best = max(rows, key=lambda row: row["mean_f1"])   # first maximum = lowest threshold
    return best["threshold"], pd.DataFrame(rows)


def threshold_curve(matches: dict[str, ClassMatch], neg_by_class: np.ndarray) -> pd.DataFrame:
    """Recall, precision, F1 and false-alarm rate at every swept threshold (test partition).

    Matching is done once; each threshold only filters by score. As in the original analysis,
    the float32 scores are compared with the threshold in double precision here.
    """
    n_neg = len(neg_by_class)
    neg64 = neg_by_class.astype(np.float64)
    rows = []
    for t in THRESHOLD_SWEEP:
        t64 = float(t)
        rec, pre = [], []
        for c in CLASSES:
            m = matches[c]
            if m.n_gt == 0:
                continue
            selected = m.scores.astype(np.float64) >= t64
            tp_n = float(m.tp[selected].sum())
            fp_n = float(selected.sum() - tp_n)
            rec.append(tp_n / m.n_gt)
            pre.append(tp_n / max(1e-9, tp_n + fp_n))
        alarmed = int((neg64 >= t64).any(axis=1).sum()) if n_neg else 0
        r, p = float(np.mean(rec)), float(np.mean(pre))
        rows.append({"threshold": round(t64, 2), "recall": round(r, 5), "precision": round(p, 5),
                     "f1": round(2 * p * r / max(1e-9, p + r), 5),
                     "far_per_1000": round(1000.0 * alarmed / max(1, n_neg), 3),
                     "n_negatives_alarmed": alarmed})
    return pd.DataFrame(rows)


def recall_and_far(matches: dict[str, ClassMatch], neg_by_class: np.ndarray, threshold: float) -> tuple[float, float]:
    """Mean recall and false-alarm fraction at one threshold (float32 comparison), for the cost model."""
    t32 = np.float32(threshold)
    rec = []
    for c in CLASSES:
        m = matches[c]
        if m.n_gt == 0:
            continue
        rec.append(float(m.tp[m.scores >= t32].sum()) / m.n_gt)
    far = float((neg_by_class >= t32).any(axis=1).mean()) if len(neg_by_class) else 0.0
    return (float(np.mean(rec)) if rec else 0.0), far

### 3.3 Analysing one run

`analyse_run` does everything that depends on a single run: it matches the validation predictions once, selects τ, evaluates the test partition at τ (all ten IoU thresholds), and keeps the matched test detections and the per-image maximum scores on negatives, from which every later section works.

In [ ]:
@dataclass
class RunResult:
    run: Run
    threshold: float
    val_sweep: pd.DataFrame
    test: dict[str, Any]
    test_matches: dict[str, ClassMatch]
    test_neg_by_class: np.ndarray
    test_neg_any: np.ndarray
    val_matches: dict[str, ClassMatch]
    val_neg_by_class: np.ndarray
    curve: pd.DataFrame
    seconds: float


def analyse_run(run: Run) -> RunResult:
    t0 = time.time()
    gts = GT_OFFICIAL if run.partition == "official" else GT
    pred_val = PREDICTIONS[(run.run_id, "val")]
    pred_test = PREDICTIONS[(run.run_id, "test")]
    val_matches = match_at_iou(pred_val, gts["val"])
    threshold, sweep = select_threshold(val_matches)
    test_eval, test_matches = evaluate_detection(pred_test, gts["test"], threshold)
    neg_by_class, neg_any = negative_max_scores(pred_test, gts["test"])
    val_neg_by_class, _ = negative_max_scores(pred_val, gts["val"])
    curve = threshold_curve(test_matches, neg_by_class)
    return RunResult(run, threshold, sweep, test_eval, test_matches, neg_by_class, neg_any,
                     val_matches, val_neg_by_class, curve, time.time() - t0)


RESULTS: dict[str, RunResult] = {}

### 3.4 Reproduction check

This analysis code was written separately from the pipeline that trained the detectors, which creates the risk of a silent formula error. The check below requires it to reproduce five values that the training pipeline recorded for the custom detector at seed 42, within the tolerances recorded with them, and to re-select the same validation threshold. If it fails, the code here is wrong, not the reference.

In [ ]:
RESULTS[REFERENCE_RUN] = analyse_run(RUNS[REFERENCE_RUN])
_ref = RESULTS[REFERENCE_RUN]
_observed = {
    "map50": _ref.test["map50"],
    "ap_fire": _ref.test["ap50"]["fire"],
    "ap_smoke": _ref.test["ap50"]["smoke"],
    "far_per_1000": _ref.test["far_per_1000"],
    "threshold": _ref.threshold,
}
REPRODUCTION_CHECK = pd.DataFrame([
    {"quantity": k, "reference": ref, "recomputed": _observed[k],
     "abs_difference": round(abs(_observed[k] - ref), 6), "tolerance": tol,
     "status": "PASS" if abs(_observed[k] - ref) <= tol else "FAIL"}
    for k, (ref, tol) in REFERENCE_VALUES.items()
])
show(REPRODUCTION_CHECK, f"Reproduction check on {REFERENCE_RUN} ({_ref.seconds:.0f} s)")
if (REPRODUCTION_CHECK.status != "PASS").any():
    raise RuntimeError("The reproduction check failed; the metric code does not match the reference.")

## 4. Table 1: composition of the curated partitions

The partitions were built by grouping perceptual near-duplicates before splitting, so that near-identical frames cannot fall on both sides of the train/test boundary. The second table records object scale on the frozen test partition, which explains a pattern that recurs in every detector: fire boxes are far smaller than smoke boxes, and small objects are where these detectors are weakest.

In [ ]:
FINDINGS: dict[str, Any] = {}   # in-text numbers, checked against the paper in Section 14


def count_boxes(cell: Any) -> int:
    return len(_parse_boxes(cell))


def write_table(frame: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / f"{name}.csv"
    frame.to_csv(path, index=False)
    return path


_dataset_rows = []
for _split in ("train", "val", "test"):
    frame = MANIFESTS[_split]
    n_fire = frame.fire_boxes.map(count_boxes)
    n_smoke = frame.smoke_boxes.map(count_boxes)
    has_object = (n_fire + n_smoke) > 0
    _dataset_rows.append({
        "partition": _split,
        "images": len(frame),
        "images_with_objects": int(has_object.sum()),
        "verified_negatives": int((~has_object).sum()),
        "percent_negative": round(100 * float((~has_object).mean()), 1),
        "fire_boxes": int(n_fire.sum()),
        "smoke_boxes": int(n_smoke.sum()),
        "images_with_fire": int((n_fire > 0).sum()),
        "images_with_smoke": int((n_smoke > 0).sum()),
        "median_width": int(frame.width.median()),
        "median_height": int(frame.height.median()),
        "manifest_hash_prefix": registry[f"dfire_{_split}"]["content_sha256"][:12],
        "frozen": bool(registry[f"dfire_{_split}"].get("immutable", False)),
    })
EXT_DATASET = pd.DataFrame(_dataset_rows)
write_table(EXT_DATASET, "ext_dataset")

_scale_rows = []
for _split in ("train", "test"):
    for column, cls in (("fire_boxes", "fire"), ("smoke_boxes", "smoke")):
        areas = pd.Series([b["w"] * b["h"] for cell in MANIFESTS[_split][column] for b in _parse_boxes(cell)])
        _scale_rows.append({
            "partition": _split, "class": cls, "n_boxes": len(areas),
            "median_area_fraction": round(float(areas.median()), 5),
            "mean_area_fraction": round(float(areas.mean()), 5),
            "percent_below_1pct_area": round(100 * float((areas < 0.01).mean()), 1),
        })
EXT_OBJECT_SCALE = pd.DataFrame(_scale_rows)
write_table(EXT_OBJECT_SCALE, "ext_object_scale")

TABLE1 = EXT_DATASET[["partition", "images", "images_with_objects", "verified_negatives",
                      "fire_boxes", "smoke_boxes"]].copy()
TABLE1.loc[TABLE1.partition == "test", "partition"] = "test (frozen)"
write_table(TABLE1, "Table1_dataset")
show(TABLE1, "Table 1. Audited composition of the curated partitions")
show(EXT_OBJECT_SCALE[EXT_OBJECT_SCALE.partition == "test"], "Object scale on the frozen test partition")

FINDINGS["total_images"] = int(EXT_DATASET.images.sum())
FINDINGS["total_verified_negatives"] = int(EXT_DATASET.verified_negatives.sum())
FINDINGS["total_fire_boxes"] = int(EXT_DATASET.fire_boxes.sum())
FINDINGS["total_smoke_boxes"] = int(EXT_DATASET.smoke_boxes.sum())
_test_scale = EXT_OBJECT_SCALE[EXT_OBJECT_SCALE.partition == "test"].set_index("class")
FINDINGS["median_fire_area_test"] = float(_test_scale.loc["fire", "median_area_fraction"])
FINDINGS["median_smoke_area_test"] = float(_test_scale.loc["smoke", "median_area_fraction"])
FINDINGS["pct_fire_boxes_below_1pct"] = float(_test_scale.loc["fire", "percent_below_1pct_area"])
FINDINGS["pct_smoke_boxes_below_1pct"] = float(_test_scale.loc["smoke", "percent_below_1pct_area"])
FINDINGS["official_partition_sizes"] = f"{OFFICIAL_COUNTS['train']} / {OFFICIAL_COUNTS['val']} / {OFFICIAL_COUNTS['test']}"
FINDINGS["official_test_images_in_curated_train"] = LEAKED_OFFICIAL_TEST_IN_CURATED_TRAIN

## 5. Evaluating every run

Each of the fourteen runs is analysed with `analyse_run` (Section 3.3): threshold selected on validation, test metrics at that threshold, threshold curves on test. This is the slow step; the custom detector's files dominate the time because it stores several hundred low-confidence boxes per image.

In [ ]:
for _run in RUNS.values():
    if _run.run_id not in RESULTS:
        RESULTS[_run.run_id] = analyse_run(_run)
    _r = RESULTS[_run.run_id]
    print(f"{_run.run_id:26s} tau {_r.threshold:.2f} | mAP50 {_r.test['map50']:.5f} | "
          f"FAR {_r.test['far_per_1000']:7.3f}/1000 | {_r.seconds:5.1f} s")

## 6. Table 3 and Fig. 1: four detectors on one frozen partition

The per-run table below is the basis of almost everything that follows. Table 3 of the paper summarises it as mean ± sample standard deviation over the three seeds.

The paper states that fire AP is below smoke AP by 0.123 to 0.169. That range is over the three-seed means of the four detectors; over the twelve single runs, all of which have fire below smoke, the gap ranges from 0.113 to 0.171. Both ranges are printed below.

In [ ]:
def per_seed_table(runs: list[Run]) -> pd.DataFrame:
    rows = []
    for run in runs:
        r = RESULTS[run.run_id].test
        rows.append({
            "detector": run.detector, "seed": run.seed, "threshold": RESULTS[run.run_id].threshold,
            "map50": r["map50"], "map50_95": r["map50_95"],
            "ap_fire": r["ap50"]["fire"], "ap_smoke": r["ap50"]["smoke"],
            "precision": r["mean_precision"], "recall": r["mean_recall"], "f1": r["mean_f1"],
            "far_per_1000": r["far_per_1000"], "n_negatives": r["n_negatives"],
            "n_negatives_alarmed": r["n_negatives_alarmed"],
        })
    return pd.DataFrame(rows).sort_values(["detector", "seed"]).reset_index(drop=True)


SUMMARY_COLUMNS = ["map50", "map50_95", "ap_fire", "ap_smoke", "precision", "recall", "f1",
                   "far_per_1000", "threshold"]

PER_SEED = per_seed_table(CURATED_RUNS)
SUMMARY = PER_SEED.groupby("detector").agg(
    **{f"{k}_{s}": (k, s) for k in SUMMARY_COLUMNS for s in ("mean", "std")},
    n_seeds=("seed", "count"),
).reset_index()
write_table(PER_SEED, "ext01_per_seed")
write_table(SUMMARY, "ext02_summary")
show(PER_SEED, "Per-run test results at the validation-selected threshold")

In [ ]:
def mean_sd(detector: str, column: str, digits: int) -> str:
    row = SUMMARY.set_index("detector").loc[detector]
    return f"{row[column + '_mean']:.{digits}f} ± {row[column + '_std']:.{digits}f}"


def threshold_range(detector: str) -> str:
    t = PER_SEED[PER_SEED.detector == detector].threshold
    return f"{t.min():.2f}" if t.min() == t.max() else f"{t.min():.2f}–{t.max():.2f}"


_table3_rows = [("τ (val-selected)", {d: threshold_range(d) for d in DETECTORS})]
for _label, _col, _digits in (("mAP50", "map50", 4), ("mAP50-95", "map50_95", 4),
                              ("AP fire", "ap_fire", 4), ("AP smoke", "ap_smoke", 4),
                              ("Precision", "precision", 4), ("Recall", "recall", 4),
                              ("FAR / 1,000", "far_per_1000", 2)):
    _table3_rows.append((_label, {d: mean_sd(d, _col, _digits) for d in DETECTORS}))
TABLE3 = pd.DataFrame([{"quantity": q, **vals} for q, vals in _table3_rows])
write_table(TABLE3, "Table3_main_results")
show(TABLE3, "Table 3. Test-partition results at each run's validation-selected threshold (mean ± SD, three seeds)")

_s = SUMMARY.set_index("detector")
FINDINGS["safest_detector_far"] = float(_s.far_per_1000_mean.min())
FINDINGS["other_detectors_far_min"] = float(_s.far_per_1000_mean.drop("ResNet18+FPN+FCOS").min())
FINDINGS["other_detectors_far_max"] = float(_s.far_per_1000_mean.drop("ResNet18+FPN+FCOS").max())
FINDINGS["alarm_gap_fold_at_operating_point"] = float(_s.far_per_1000_mean.max() / _s.far_per_1000_mean.min())
FINDINGS["custom_recall_mean"] = float(_s.loc["ResNet18+FPN+FCOS", "recall_mean"])
FINDINGS["most_accurate_recall_mean"] = float(_s.loc[_s.map50_mean.idxmax(), "recall_mean"])
FINDINGS["yolo_map50_min"] = float(_s.map50_mean.drop("ResNet18+FPN+FCOS").min())
FINDINGS["yolo_map50_max"] = float(_s.map50_mean.drop("ResNet18+FPN+FCOS").max())
FINDINGS["custom_map50_gap_min_points"] = 100 * (FINDINGS["yolo_map50_min"] - _s.loc["ResNet18+FPN+FCOS", "map50_mean"])
FINDINGS["custom_map50_gap_max_points"] = 100 * (FINDINGS["yolo_map50_max"] - _s.loc["ResNet18+FPN+FCOS", "map50_mean"])
_gap_runs = PER_SEED.ap_smoke - PER_SEED.ap_fire
_gap_means = _s.ap_smoke_mean - _s.ap_fire_mean
FINDINGS["runs_with_fire_ap_below_smoke"] = int((_gap_runs > 0).sum())
FINDINGS["fire_smoke_gap_detector_means_min"] = float(_gap_means.min())
FINDINGS["fire_smoke_gap_detector_means_max"] = float(_gap_means.max())
FINDINGS["fire_smoke_gap_single_runs_min"] = float(_gap_runs.min())
FINDINGS["fire_smoke_gap_single_runs_max"] = float(_gap_runs.max())
FINDINGS["map50_seed_sd_min"] = float(_s.map50_std.min())
FINDINGS["map50_seed_sd_max"] = float(_s.map50_std.max())
FINDINGS["far_seed_sd_min"] = float(_s.far_per_1000_std.min())
FINDINGS["far_seed_sd_max"] = float(_s.far_per_1000_std.max())
FINDINGS["verified_negatives_test"] = GT["test"].n_negatives
print("Fire AP is below smoke AP in", FINDINGS["runs_with_fire_ap_below_smoke"], "of 12 runs; the gap is "
      f"{FINDINGS['fire_smoke_gap_detector_means_min']:.3f} to {FINDINGS['fire_smoke_gap_detector_means_max']:.3f} "
      f"over detector means and {FINDINGS['fire_smoke_gap_single_runs_min']:.3f} to "
      f"{FINDINGS['fire_smoke_gap_single_runs_max']:.3f} over single runs.")

**Fig. 1** plots mAP50 against the false-alarm rate for all twelve runs on a logarithmic vertical axis: hollow markers are single seeds, solid markers three-seed means. The figure style reproduces the published one.

In [ ]:
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300,
    "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
    "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})
MARKER = {"ResNet18+FPN+FCOS": "D", "YOLO11n": "o", "YOLO11s": "s", "YOLOv8s": "^"}
COLOUR = {"ResNet18+FPN+FCOS": "#1b1b1b", "YOLO11n": "#4c72b0", "YOLO11s": "#dd8452", "YOLOv8s": "#55a868"}


def finish_figure(fig: plt.Figure, name: str) -> None:
    """Save a figure under outputs/figures, show it inline when running in Jupyter, then close it."""
    path = FIGURE_DIR / f"{name}.png"
    fig.savefig(path)
    try:
        get_ipython  # noqa: B018  (only defined inside IPython)
        from IPython.display import display

        display(fig)
    except NameError:
        pass
    plt.close(fig)


fig, ax = plt.subplots(figsize=(4.8, 3.6))
for det in SUMMARY.detector:
    sub = PER_SEED[PER_SEED.detector == det]
    ax.scatter(sub.map50, sub.far_per_1000, marker=MARKER[det], s=26, facecolors="none",
               edgecolors=COLOUR[det], linewidths=1.0, zorder=2)
    row = SUMMARY[SUMMARY.detector == det].iloc[0]
    ax.scatter([row.map50_mean], [row.far_per_1000_mean], marker=MARKER[det], s=95, color=COLOUR[det],
               edgecolors="black", linewidths=0.6, zorder=3, label=det)
ax.set_yscale("log")
ax.set_xlabel("mAP50 (test split)")
ax.set_ylabel("False alarms per 1,000 verified negatives\n(log scale)")
ax.set_title("Hollow = individual seed · solid = three-seed mean")
ax.legend(frameon=False, loc="lower right", fontsize=7.5)
finish_figure(fig, "Fig1")

## 7. Table 4: false alarms at equal recall

A detector tuned to a higher recall should be expected to fire more often, so part of the gap in Table 3 comes from where each validation sweep placed the threshold. Here every run's test curve (recall and false-alarm rate at each swept threshold) is interpolated linearly at fixed recalls. A recall outside the range a run reaches is left empty rather than extrapolated. The multipliers are relative to the custom detector at the same recall.

In [ ]:
def far_at_recall(curve: pd.DataFrame, target: float) -> float | None:
    ordered = curve.sort_values("recall")
    if target < ordered.recall.min() or target > ordered.recall.max():
        return None
    return round(float(np.interp(target, ordered.recall.values, ordered.far_per_1000.values)), 3)


_rows = []
for run in CURATED_RUNS:
    curve = RESULTS[run.run_id].curve
    row = {"detector": run.detector, "seed": run.seed}
    for target in FIXED_RECALLS:
        row[f"far_at_recall_{target}"] = far_at_recall(curve, target)
    row["highest_recall"] = round(float(curve.recall.max()), 5)
    _rows.append(row)
FAR_AT_RECALL = pd.DataFrame(_rows).sort_values(["detector", "seed"]).reset_index(drop=True)
FAR_AT_RECALL_MEAN = FAR_AT_RECALL.groupby("detector").mean(numeric_only=True).drop(columns=["seed"]).reset_index()
write_table(FAR_AT_RECALL, "ext03_far_at_recall_per_seed")
write_table(FAR_AT_RECALL_MEAN, "ext03b_far_at_recall_mean")

_m = FAR_AT_RECALL_MEAN.set_index("detector")
_ratios = []
_table4_rows = []
for det in DETECTORS:
    row = {"detector": det}
    for target in FIXED_RECALLS:
        value = _m.loc[det, f"far_at_recall_{target}"]
        if det == "ResNet18+FPN+FCOS":
            row[f"R = {target}"] = f"{value:.2f}"
        else:
            ratio = value / _m.loc["ResNet18+FPN+FCOS", f"far_at_recall_{target}"]
            _ratios.append(ratio)
            row[f"R = {target}"] = f"{value:.2f} ({ratio:.2f}x)"
    row["highest recall attained"] = f"{_m.loc[det, 'highest_recall']:.4f}"
    _table4_rows.append(row)
TABLE4 = pd.DataFrame(_table4_rows)
write_table(TABLE4, "Table4_far_at_fixed_recall")
show(TABLE4, "Table 4. False alarms per 1,000 verified negatives at fixed recall, three-seed means")
FINDINGS["matched_recall_ratio_min"] = float(min(_ratios))
FINDINGS["matched_recall_ratio_max"] = float(max(_ratios))
FINDINGS["custom_highest_recall"] = float(_m.loc["ResNet18+FPN+FCOS", "highest_recall"])
FINDINGS["yolo_highest_recall_min"] = float(_m.highest_recall.drop("ResNet18+FPN+FCOS").min())
FINDINGS["yolo_highest_recall_max"] = float(_m.highest_recall.drop("ResNet18+FPN+FCOS").max())

## 8. Table 5: which differences are statistically real

**Seed-averaged paired bootstrap.** For every verified-negative test image, each detector's three runs either raise an alarm or not, each at its own validation-selected threshold. Averaging the three indicators gives that detector's alarm probability on that image. These per-image probabilities are paired across detectors (same images) and resampled 2,000 times with replacement. The two-sided *p* is twice the share of resamples whose mean difference has the opposite sign to the observed one. The resampling is over test images, so the intervals measure dependence on the test images; seed variation enters through the averaging, not through the resampling, because three seeds are too few to resample.

The same test on seed 42 alone is the last column of Table 5. It is what a one-run-per-model study would report.

The random generator is `numpy.random.default_rng(42)`, drawn one resample at a time exactly as in the original analysis, so the intervals are reproduced to the last digit.

In [ ]:
def bootstrap_mean(values: np.ndarray, n_resamples: int = BOOTSTRAP_RESAMPLES,
                   level: float = CONFIDENCE_LEVEL, seed: int = RANDOM_SEED) -> dict[str, float]:
    x = np.asarray(values, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = np.empty(n_resamples, dtype=np.float64)
    for i in range(n_resamples):
        means[i] = x[rng.integers(0, len(x), len(x))].mean()
    tail = (1.0 - level) / 2.0
    return {"mean": float(x.mean()), "lower": float(np.quantile(means, tail)),
            "upper": float(np.quantile(means, 1.0 - tail)), "n": int(len(x))}


def paired_bootstrap(a: np.ndarray, b: np.ndarray, n_resamples: int = BOOTSTRAP_RESAMPLES,
                     level: float = CONFIDENCE_LEVEL, seed: int = RANDOM_SEED) -> dict[str, float]:
    diff = np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = np.empty(n_resamples, dtype=np.float64)
    for i in range(n_resamples):
        means[i] = diff[rng.integers(0, len(diff), len(diff))].mean()
    tail = (1.0 - level) / 2.0
    observed = float(diff.mean())
    flipped = float((np.sign(means) != np.sign(observed)).mean()) if observed != 0 else 1.0
    return {"difference": observed, "lower": float(np.quantile(means, tail)),
            "upper": float(np.quantile(means, 1.0 - tail)), "p": min(1.0, 2.0 * flipped),
            "n": int(len(diff))}


def alarm_indicator(run: Run) -> np.ndarray:
    """1 where a verified-negative test image raises an alarm at the run's threshold, else 0."""
    r = RESULTS[run.run_id]
    return (r.test_neg_any >= float(r.threshold)).astype(np.float64)


def bootstrap_table(per_detector: dict[str, np.ndarray], kind: str, n_seeds: dict[str, int] | None) -> pd.DataFrame:
    rows = []
    for det, v in per_detector.items():
        ci = bootstrap_mean(v * 1000.0)
        row = {"kind": f"bootstrap interval of FAR{kind}", "a": det, "b": "",
               "value": round(ci["mean"], 3), "lower": round(ci["lower"], 3),
               "upper": round(ci["upper"], 3), "p": None, "n": ci["n"]}
        if n_seeds is not None:
            row["n_seeds"] = n_seeds[det]
        rows.append(row)
    for a, b in itertools.combinations(per_detector, 2):
        t = paired_bootstrap(per_detector[a] * 1000.0, per_detector[b] * 1000.0)
        row = {"kind": f"paired test of FAR difference{kind}", "a": a, "b": b,
               "value": round(t["difference"], 3), "lower": round(t["lower"], 3),
               "upper": round(t["upper"], 3), "p": round(t["p"], 4), "n": t["n"]}
        if n_seeds is not None:
            row["n_seeds"] = min(n_seeds[a], n_seeds[b])
        rows.append(row)
    return pd.DataFrame(rows)


_seed42 = {r.detector: alarm_indicator(r) for r in CURATED_RUNS if r.seed == 42}
_by_detector: dict[str, list[np.ndarray]] = {}
for _run in CURATED_RUNS:
    _by_detector.setdefault(_run.detector, []).append(alarm_indicator(_run))
_seed_avg = {det: np.mean(np.vstack(v), axis=0) for det, v in _by_detector.items()}

BOOT_SEED42 = bootstrap_table(_seed42, "", None)
BOOT_SEED_AVG = bootstrap_table(_seed_avg, " (seed-averaged)", {d: len(v) for d, v in _by_detector.items()})
write_table(BOOT_SEED42, "ext09_bootstrap_seed42")
write_table(BOOT_SEED_AVG, "ext09b_bootstrap_seed_averaged")


def format_p(p: float) -> str:
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def half_up(x: float, digits: int = 2) -> str:
    """Round a stored three-decimal value half away from zero, as the paper's tables do (4.885 -> 4.89)."""
    return str(Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-digits), rounding=ROUND_HALF_UP))


_table5 = []
_avg = BOOT_SEED_AVG.set_index(["a", "b"])
_s42 = BOOT_SEED42.set_index(["a", "b"])
for det in DETECTORS:
    r, q = _avg.loc[(det, "")], _s42.loc[(det, "")]
    _table5.append({"comparison": det, "FAR or difference / 1,000": half_up(r.value),
                    "95 % interval": f"[{half_up(r.lower)}, {half_up(r.upper)}]", "p (two-sided)": "---",
                    "same test, seed 42 alone": half_up(q.value)})
for a, b in itertools.combinations(DETECTORS, 2):
    r, q = _avg.loc[(a, b)], _s42.loc[(a, b)]
    _table5.append({"comparison": f"{SHORT_NAME[a]} - {SHORT_NAME[b]}",
                    "FAR or difference / 1,000": half_up(r.value),
                    "95 % interval": f"[{half_up(r.lower)}, {half_up(r.upper)}]", "p (two-sided)": format_p(r.p),
                    "same test, seed 42 alone": f"{half_up(q.value)} ({format_p(q.p)})"})
TABLE5 = pd.DataFrame(_table5)
write_table(TABLE5, "Table5_paired_comparisons")
show(TABLE5, "Table 5. Paired comparisons over the shared verified negatives, seed-averaged; last column seed 42 alone")

_pairs_avg = BOOT_SEED_AVG[BOOT_SEED_AVG.b != ""]
_pairs_42 = BOOT_SEED42[BOOT_SEED42.b != ""]
FINDINGS["separable_pairs_seed_averaged"] = int((_pairs_avg.p < 0.05).sum())
FINDINGS["separable_pairs_seed42"] = int((_pairs_42.p < 0.05).sum())
FINDINGS["separable_pairs_seed_averaged_by_interval"] = int(((_pairs_avg.lower > 0) | (_pairs_avg.upper < 0)).sum())
print(f"Separable pairs (p < 0.05): {FINDINGS['separable_pairs_seed_averaged']} of 6 seed-averaged, "
      f"{FINDINGS['separable_pairs_seed42']} of 6 on seed 42 alone.")

## 9. Rank correlation: two ways, 81 single-seed studies, and the exact null

Spearman's coefficient is computed between the order by mAP50 (best first) and the order by false-alarm rate (safest first), so that agreement gives a positive value. With four detectors and no ties,

$$\rho = 1 - \frac{6\sum d^2}{n(n^2-1)}, \qquad n = 4.$$

It is computed in three ways:

1. from the seed-42 runs alone, as a one-run-per-model study would;
2. from three-seed means;
3. for every one of the $3^4 = 81$ ways to pick one run per detector, each of which is a complete single-seed study.

Because $n = 4$, the null distribution of $\rho$ needs no approximation: there are $4! = 24$ ways to pair one ranking with the other, and the exact two-sided *p*-value is the share of those pairings whose $|\rho|$ is at least as large as the observed one. The same enumeration for $n = 3, \dots, 10$ gives the smallest *p*-value any rank correlation can reach with *n* detectors (Table 6). That table depends on no measurement at all; it is a property of the design.

In [ ]:
def spearman_from_orders(order_a: list[str], order_b: list[str]) -> float:
    """Spearman's rho for two complete orderings without ties (lists of names, best first)."""
    n = len(order_a)
    rank_a = {m: i + 1 for i, m in enumerate(order_a)}
    rank_b = {m: i + 1 for i, m in enumerate(order_b)}
    d2 = sum((rank_a[m] - rank_b[m]) ** 2 for m in order_a)
    return 1.0 - (6.0 * d2) / (n * (n ** 2 - 1))


def spearman_from_values(higher_is_better: dict[str, float], lower_is_better: dict[str, float]) -> float:
    """rho between the order by the first quantity (descending) and by the second (ascending)."""
    names = list(higher_is_better)
    order_a = sorted(names, key=lambda m: -higher_is_better[m])
    order_b = sorted(names, key=lambda m: lower_is_better[m])
    return spearman_from_orders(order_a, order_b)


def exact_spearman_null(n: int) -> np.ndarray:
    """rho for every one of the n! pairings of two rankings of n items (complete enumeration)."""
    count = math.factorial(n)
    perms = np.fromiter(itertools.chain.from_iterable(itertools.permutations(range(n))),
                        dtype=np.int8, count=n * count).reshape(count, n).astype(np.int64)
    d2 = ((perms - np.arange(n)) ** 2).sum(axis=1)
    return 1.0 - 6.0 * d2 / (n * (n * n - 1))


def exact_p_two_sided(rho: float, null: np.ndarray) -> float:
    return float((np.abs(null) >= abs(rho) - 1e-9).mean())


# (a) one seed per detector, as a single-run study would do it
_s42 = PER_SEED[PER_SEED.seed == 42].set_index("detector")
RHO_SEED42 = spearman_from_values(_s42.map50.to_dict(), _s42.far_per_1000.to_dict())
# (b) three-seed means
MAP50_MEAN = dict(zip(SUMMARY.detector, SUMMARY.map50_mean))
FAR_MEAN = dict(zip(SUMMARY.detector, SUMMARY.far_per_1000_mean))
RHO_MEANS = spearman_from_values(MAP50_MEAN, FAR_MEAN)

RANK_TWO_WAYS = pd.DataFrame([
    {"method": "single seed (42 only)", "rho": round(RHO_SEED42, 4),
     "map50_order": " > ".join(sorted(_s42.index, key=lambda m: -_s42.map50[m])),
     "far_order_safest_first": " > ".join(_s42.far_per_1000.sort_values().index.tolist())},
    {"method": "three-seed means", "rho": round(RHO_MEANS, 4),
     "map50_order": " > ".join(sorted(MAP50_MEAN, key=lambda m: -MAP50_MEAN[m])),
     "far_order_safest_first": " > ".join(sorted(FAR_MEAN, key=lambda m: FAR_MEAN[m]))},
])
write_table(RANK_TWO_WAYS, "ext06_rank_correlation_two_ways")
show(RANK_TWO_WAYS, "Spearman rho between mAP50 order and false-alarm order")

# (c) all 3^4 = 81 single-seed studies
_detectors = SUMMARY.detector.tolist()
_lookup = {(r.detector, r.seed): r for r in PER_SEED.itertuples(index=False)}
_rho_all, _combo_rows = [], []
for combo in itertools.product(SEEDS, repeat=len(_detectors)):
    m = {d: _lookup[(d, s)].map50 for d, s in zip(_detectors, combo)}
    f = {d: _lookup[(d, s)].far_per_1000 for d, s in zip(_detectors, combo)}
    rho = spearman_from_values(m, f)
    _rho_all.append(rho)
    _combo_rows.append({**{f"seed_{d}": s for d, s in zip(_detectors, combo)}, "rho": round(rho, 4)})
RHO_81 = np.asarray(_rho_all)
RHO_COMBINATIONS = pd.DataFrame(_combo_rows)
RHO_DISTRIBUTION = pd.DataFrame([{
    "n_combinations": len(RHO_81),
    "rho_mean": round(float(RHO_81.mean()), 4),
    "rho_sd": round(float(RHO_81.std(ddof=1)), 4),
    "rho_min": round(float(RHO_81.min()), 4),
    "rho_q1": round(float(np.quantile(RHO_81, 0.25)), 4),
    "rho_median": round(float(np.median(RHO_81)), 4),
    "rho_q3": round(float(np.quantile(RHO_81, 0.75)), 4),
    "rho_max": round(float(RHO_81.max()), 4),
    "share_rho_le_-0.5": round(float((RHO_81 <= -0.5).mean()), 4),
    "share_rho_le_-0.8": round(float((RHO_81 <= -0.8).mean()), 4),
    "share_rho_ge_0": round(float((RHO_81 >= 0).mean()), 4),
    "rho_three_seed_means": round(RHO_MEANS, 4),
}])
write_table(RHO_COMBINATIONS, "ext07b_rho_81_combinations")
write_table(RHO_DISTRIBUTION, "ext07_rho_distribution")
_values, _counts = np.unique(np.round(RHO_81, 4), return_counts=True)
show(pd.DataFrame({"rho": _values, "single-seed studies": _counts}), "Distribution of rho over the 81 single-seed studies")
show(RHO_DISTRIBUTION, "Summary of the 81 values")

# (d) exact permutation tests at n = 4, and the floor table for n = 3..10
NULL_4 = exact_spearman_null(4)
PERMUTATION_TESTS = pd.DataFrame([
    {"case": case, "rho": round(value, 4), "p_two_sided_exact": round(exact_p_two_sided(value, NULL_4), 4),
     "n_permutations": len(NULL_4), "significant_at_0.05": bool(exact_p_two_sided(value, NULL_4) < 0.05)}
    for case, value in (("single seed (42 only)", RHO_SEED42), ("three-seed means", RHO_MEANS),
                        ("most extreme rho possible", -1.0))
])
write_table(PERMUTATION_TESTS, "ext08_exact_permutation_tests")
show(PERMUTATION_TESTS, "Exact permutation null at n = 4 (all 24 pairings)")

_floor_rows = []
for n in range(3, 11):
    null = exact_spearman_null(n)
    p_min = exact_p_two_sided(1.0, null)   # a perfect agreement or reversal, |rho| = 1
    _floor_rows.append({"n_detectors": n, "n_permutations": len(null), "most_extreme_abs_rho": 1.0,
                        "smallest_two_sided_p": round(p_min, 5), "can_reach_0.05": bool(p_min < 0.05)})
PERMUTATION_FLOOR = pd.DataFrame(_floor_rows)
write_table(PERMUTATION_FLOOR, "ext08b_permutation_floor")


def format_floor_p(p: float) -> str:
    """Three decimals down to 0.01, two significant digits below, as printed in the paper."""
    if p >= 0.01:
        return f"{p:.3f}"
    return np.format_float_positional(float(f"{p:.2g}"), trim="-")


TABLE6 = pd.DataFrame([
    {"detectors": int(r.n_detectors), "rank pairings": int(r.n_permutations),
     "smallest p": format_floor_p(r.smallest_two_sided_p), "can reach 0.05": "yes" if r["can_reach_0.05"] else "no"}
    for _, r in PERMUTATION_FLOOR.iterrows() if r.n_detectors in (3, 4, 5, 6, 8)
])
write_table(TABLE6, "Table6_permutation_floor")
show(TABLE6, "Table 6. Smallest two-sided p a rank correlation can attain, by number of detectors")

FINDINGS["rho_seed42"] = RHO_SEED42
FINDINGS["rho_three_seed_means"] = RHO_MEANS
for _v, _c in zip(_values, _counts):
    FINDINGS[f"draws_with_rho_{_v:+.2f}"] = int(_c)
FINDINGS["rho_81_mean"] = float(RHO_81.mean())
FINDINGS["rho_81_sd"] = float(RHO_81.std(ddof=1))
FINDINGS["rho_81_share_strongest_pct"] = 100 * float((RHO_81 <= RHO_81.min() + 1e-9).mean())
FINDINGS["rho_81_draws_non_negative"] = int((RHO_81 >= 0).sum())
FINDINGS["rho_81_magnitude_ratio"] = float(np.abs(RHO_81).max() / np.abs(RHO_81).min())
FINDINGS["exact_p_seed42"] = exact_p_two_sided(RHO_SEED42, NULL_4)
FINDINGS["exact_p_three_seed_means"] = exact_p_two_sided(RHO_MEANS, NULL_4)
FINDINGS["exact_p_floor_n4"] = exact_p_two_sided(1.0, NULL_4)
FINDINGS["exact_p_floor_n3"] = exact_p_two_sided(1.0, exact_spearman_null(3))

## 10. Table 8: expected cost, and a boundary artefact

A miss and a false alarm do not cost the same. With the false-alarm cost fixed at 1 and $r = C_{\text{miss}} / C_{\text{false alarm}}$,

$$C_{\text{exp}} = r\,(1 - \text{Recall}) + \text{FAR},$$

where FAR is the per-image alarm fraction (the rate per 1,000 divided by 1,000). For each run and each $r$, the threshold that minimises the cost on **validation** is found by the same sweep, then applied once to test. The ratios are sensitivity parameters, not costs reported by any fire agency.

Because predictions were stored down to 0.05, the sweep cannot go lower. A cost optimum that lands exactly on 0.05 is a boundary solution whose true optimum may lie below it; such cells are flagged rather than hidden.

In [ ]:
_cost_rows = []
for run in CURATED_RUNS:
    r = RESULTS[run.run_id]
    for ratio in COST_RATIOS:
        best_t, best_cost = None, float("inf")
        for t in THRESHOLD_SWEEP:
            rec, far = recall_and_far(r.val_matches, r.val_neg_by_class, float(t))
            cost = ratio * (1.0 - rec) + far
            if cost < best_cost:        # strict: ties keep the lower threshold
                best_cost, best_t = cost, float(t)
        rec_t, far_t = recall_and_far(r.test_matches, r.test_neg_by_class, best_t)
        _cost_rows.append({
            "detector": run.detector, "seed": run.seed, "cost_ratio": ratio,
            "threshold_selected_on_val": round(best_t, 2),
            "recall_test": round(rec_t, 5), "far_test": round(far_t, 5),
            "cost_test": round(ratio * (1.0 - rec_t) + far_t, 5),
            "at_sweep_floor": bool(abs(best_t - float(THRESHOLD_SWEEP[0])) < 1e-9),
        })
COST = pd.DataFrame(_cost_rows)
write_table(COST, "ext04_expected_cost_per_seed")

_cost_corr = []
for ratio in COST_RATIOS:
    block = COST[COST.cost_ratio == ratio]
    mean_cost = block.groupby("detector").cost_test.mean()
    at_floor = sorted(block[block.at_sweep_floor].detector.unique())
    _cost_corr.append({
        "cost_ratio": ratio,
        "rho_map50_vs_cost": round(spearman_from_values(MAP50_MEAN, mean_cost.to_dict()), 4),
        "map50_order": " > ".join(sorted(MAP50_MEAN, key=lambda m: -MAP50_MEAN[m])),
        "cost_order_cheapest_first": " > ".join(mean_cost.sort_values().index.tolist()),
        "cheapest": mean_cost.idxmin(),
        "best_map50": max(MAP50_MEAN, key=MAP50_MEAN.get),
        "detectors_at_sweep_floor": "; ".join(at_floor) if at_floor else "(none)",
    })
COST_CORRELATION = pd.DataFrame(_cost_corr)
write_table(COST_CORRELATION, "ext05_cost_rank_correlation")


def floor_label(detectors: list[str]) -> str:
    yolo = {d for d in DETECTORS if FAMILY[d] == "YOLO"}
    if set(detectors) == set(DETECTORS):
        return "all 4"
    if set(detectors) == yolo:
        return "3 YOLO"
    return ", ".join(SHORT_NAME[d] for d in detectors) or "none"


_table8 = []
for ratio in COST_RATIOS:
    block = COST[COST.cost_ratio == ratio]
    mean_cost = block.groupby("detector").cost_test.mean()
    corr = COST_CORRELATION[COST_CORRELATION.cost_ratio == ratio].iloc[0]
    _table8.append({
        "r": ratio, **{SHORT_NAME[d]: f"{mean_cost[d]:.3f}" for d in DETECTORS},
        "cheapest": SHORT_NAME[corr.cheapest], "rho (mAP50 vs cost)": f"{corr.rho_map50_vs_cost:+.2f}",
        "at sweep floor": floor_label(sorted(block[block.at_sweep_floor].detector.unique())),
    })
TABLE8 = pd.DataFrame(_table8)
write_table(TABLE8, "Table8_expected_cost")
show(TABLE8, "Table 8. Expected cost at test (thresholds selected on validation), three-seed means")

_yolo_cells = COST[COST.detector.map(FAMILY) == "YOLO"]
_custom_cells = COST[COST.detector == "ResNet18+FPN+FCOS"]
FINDINGS["yolo_cells_at_sweep_floor"] = int(_yolo_cells.at_sweep_floor.sum())
FINDINGS["yolo_cells_total"] = len(_yolo_cells)
FINDINGS["yolo_cells_off_floor"] = "; ".join(
    f"{r.detector} seed {r.seed} r={r.cost_ratio}" for r in _yolo_cells[~_yolo_cells.at_sweep_floor].itertuples())
FINDINGS["custom_cells_at_sweep_floor"] = int(_custom_cells.at_sweep_floor.sum())
FINDINGS["custom_cells_at_floor_where"] = "; ".join(
    f"seed {r.seed} r={r.cost_ratio}" for r in _custom_cells[_custom_cells.at_sweep_floor].itertuples())
print(f"YOLO cells on the 0.05 floor: {FINDINGS['yolo_cells_at_sweep_floor']} of {FINDINGS['yolo_cells_total']} "
      f"(off the floor: {FINDINGS['yolo_cells_off_floor']}); custom detector: "
      f"{FINDINGS['custom_cells_at_sweep_floor']} ({FINDINGS['custom_cells_at_floor_where']}).")

## 11. Table 2, Table 7 and Fig. 2: capacity, and what predicts the alarm rate

**Table 2 is displayed, not recomputed.** Parameter counts, GFLOPs, model size, latency, FPS and peak memory were measured on the RTX 4050 Laptop GPU used for training (batch 1, 640 px, 20 warm-up and 100 timed iterations, idle GPU, decoding/letterboxing/NMS excluded). They cannot be reproduced on a CPU and are read from the measurement files in `data/recorded/`, whose provenance is described in `data/recorded/README.md`. GFLOPs for the custom detector were not measured because the profiling dependency was not installed in the training environment. The GFLOPs shown count a multiply-accumulate once; the Ultralytics model summary (`ultralytics_model_summary.txt`) prints roughly twice these figures under the opposite convention.

One display difference: the recorded checkpoint size of YOLO11n is 5.45 MB, which is printed here as 5.5 and in the paper as 5.4. Both are roundings of the same recorded value at its midpoint, and the check in Section 14 accepts either.

**Table 7** ranks the four detectors by each of six candidate predictors and correlates each ordering with the alarm-safety ordering, over all four detectors and within the YOLO family alone ($n = 3$).

In [ ]:
_efficiency = pd.read_csv(RECORDED_DIR / "efficiency_summary_server.csv")
_e0b = _efficiency[_efficiency.Model == "E0b"]
_e0b42 = _e0b[_e0b.Seed == 42].iloc[0]
_capacity_rows = [{
    "detector": "ResNet18+FPN+FCOS", "params_m": float(_e0b42["Parameters (M)"]), "gflops": None,
    "size_mb": float(_e0b42["Model size (MB)"]), "latency_ms": float(_e0b42["Latency mean (ms)"]),
    "fps": float(_e0b42["FPS"]), "peak_vram_gb": float(_e0b42["Peak VRAM (GB)"]),
    "source": "efficiency harness (E0b seed 42)",
}]
for r in pd.read_csv(RECORDED_DIR / "yolo_efficiency_benchmark.csv").itertuples(index=False):
    _capacity_rows.append({
        "detector": r.label, "params_m": round(r.params_m, 4), "gflops": r.gflops,
        "size_mb": round(r.checkpoint_size_mb, 3), "latency_ms": round(r.latency_mean_ms, 4),
        "fps": round(r.fps, 3), "peak_vram_gb": r.peak_vram_gb,
        "source": "YOLO efficiency benchmark (idle GPU)",
    })
CAPACITY = pd.DataFrame(_capacity_rows)
CAPACITY["family"] = CAPACITY.detector.map(FAMILY)
CAPACITY = CAPACITY.sort_values("detector", key=lambda s: s.map({d: i for i, d in enumerate(FAMILY)})).reset_index(drop=True)
write_table(CAPACITY, "ext11_capacity")

TABLE2 = pd.DataFrame([{
    "detector": r.detector, "params (M)": f"{r.params_m:.3f}",
    "GFLOPs": "---" if pd.isna(r.gflops) else f"{r.gflops:.2f}",
    "size (MB)": f"{r.size_mb:.1f}", "latency (ms)": f"{r.latency_ms:.2f}", "FPS": f"{r.fps:.1f}",
} for r in CAPACITY.itertuples(index=False)])
write_table(TABLE2, "Table2_capacity")
show(TABLE2, "Table 2. The four detectors and their cost at seed 42 (recorded GPU measurements, display only)")

_fastest = CAPACITY.latency_ms.min()
FINDINGS["custom_latency_ms_seed42"] = float(CAPACITY.set_index("detector").loc["ResNet18+FPN+FCOS", "latency_ms"])
FINDINGS["fastest_latency_ms"] = float(_fastest)
FINDINGS["custom_slower_than_fastest_pct"] = 100 * (FINDINGS["custom_latency_ms_seed42"] / _fastest - 1)
FINDINGS["custom_latency_ms_min_over_seeds"] = float(_e0b["Latency mean (ms)"].min())
FINDINGS["custom_latency_ms_max_over_seeds"] = float(_e0b["Latency mean (ms)"].max())
FINDINGS["custom_fps_min_over_seeds"] = float(_e0b["FPS"].min())
FINDINGS["custom_fps_max_over_seeds"] = float(_e0b["FPS"].max())

In [ ]:
_ceiling = FAR_AT_RECALL.groupby("detector").highest_recall.mean().to_dict()
CANDIDATES: dict[str, dict[str, float]] = {
    "mAP50": dict(zip(SUMMARY.detector, SUMMARY.map50_mean)),
    "mAP50-95": dict(zip(SUMMARY.detector, SUMMARY.map50_95_mean)),
    "recall at operating point": dict(zip(SUMMARY.detector, SUMMARY.recall_mean)),
    "precision at operating point": dict(zip(SUMMARY.detector, SUMMARY.precision_mean)),
    "highest recall attained": _ceiling,
}
if CAPACITY.params_m.notna().all():
    CANDIDATES["parameter count"] = dict(zip(CAPACITY.detector, CAPACITY.params_m))

_yolo = [d for d, fam in FAMILY.items() if fam == "YOLO"]
_pred_rows = []
for name, values in CANDIDATES.items():
    if set(values) != set(FAR_MEAN):
        continue
    _pred_rows.append({
        "predictor": name,
        "rho_all_four": round(spearman_from_values(values, FAR_MEAN), 4),
        "rho_yolo_only": round(spearman_from_values({d: values[d] for d in _yolo}, {d: FAR_MEAN[d] for d in _yolo}), 4),
        "predictor_order_best_first": " > ".join(sorted(values, key=lambda m: -values[m])),
        "alarm_order_safest_first": " > ".join(sorted(FAR_MEAN, key=lambda m: FAR_MEAN[m])),
    })
PREDICTORS = pd.DataFrame(_pred_rows)
write_table(PREDICTORS, "ext12_alarm_predictors")
TABLE7 = pd.DataFrame([{"predictor": r.predictor, "all four": f"{r.rho_all_four:+.2f}",
                        "YOLO only (n = 3)": f"{r.rho_yolo_only:+.2f}"} for r in PREDICTORS.itertuples()])
write_table(TABLE7, "Table7_predictors")
show(TABLE7, "Table 7. Predictors of the false-alarm ordering (rho against 'safest first')")

# Screening six predictors at n = 4: chance that at least one orders all four perfectly by luck,
# if the predictors were independent (in either direction, and in the correct direction only).
_p4 = FINDINGS["exact_p_floor_n4"]
FINDINGS["n_predictors_screened"] = len(PREDICTORS)
FINDINGS["multiplicity_any_direction"] = 1 - (1 - _p4) ** len(PREDICTORS)
FINDINGS["multiplicity_correct_direction"] = 1 - (1 - _p4 / 2) ** len(PREDICTORS)
print(f"Multiplicity over {len(PREDICTORS)} predictors: {FINDINGS['multiplicity_any_direction']:.2f} "
      f"(either direction), {FINDINGS['multiplicity_correct_direction']:.2f} (correct direction)")

**Fig. 2.** False-alarm rate against detection accuracy (a) and against model capacity (b), with three-seed error bars on both axes in (a) and on the vertical axis in (b); parameter count has no seed variance.

In [ ]:
_merged = SUMMARY.merge(CAPACITY[["detector", "params_m", "family"]], on="detector")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.9, 3.2))
for r in _merged.itertuples(index=False):
    style = dict(marker=MARKER[r.detector], s=110, color=COLOUR[r.detector], edgecolors="black",
                 linewidths=0.6, zorder=3)
    ax1.errorbar(r.map50_mean, r.far_per_1000_mean, xerr=r.map50_std, yerr=r.far_per_1000_std,
                 fmt="none", ecolor="0.6", elinewidth=0.9, capsize=2.5, zorder=2)
    ax1.scatter(r.map50_mean, r.far_per_1000_mean, **style)
    ax2.errorbar(r.params_m, r.far_per_1000_mean, yerr=r.far_per_1000_std,
                 fmt="none", ecolor="0.6", elinewidth=0.9, capsize=2.5, zorder=2)
    ax2.scatter(r.params_m, r.far_per_1000_mean, label=r.detector, **style)
# Dashed line in parameter order, to make the monotone relation visible.
_ordered = _merged.sort_values("params_m")
ax2.plot(_ordered.params_m, _ordered.far_per_1000_mean, color="0.45", linewidth=1.0, linestyle="--", zorder=1)
ax1.set_xlabel("mAP50 (three-seed mean)")
ax1.set_ylabel("False alarms / 1,000 verified negatives")
ax1.set_title("(a) against detection accuracy", fontsize=9)
ax2.set_xlabel("Trainable parameters (millions)")
ax2.set_title("(b) against model capacity", fontsize=9)
ax2.legend(frameon=False, fontsize=7.2, loc="lower left")
for _ax in (ax1, ax2):
    _ax.set_ylim(0, 42)
finish_figure(fig, "Fig2")

## 12. Split influence: YOLO11n retrained on the official D-Fire partition

Our deduplicated partition mixes D-Fire's train and test folders, so our numbers cannot be set beside published D-Fire figures directly. To isolate partition choice, YOLO11n was retrained from scratch on the official folders (Section 2.3) with every hyperparameter unchanged, and evaluated on the full official test folder. Its threshold is selected on the official validation split exactly as for the other runs. Two seeds were run.

In [ ]:
_y = SUMMARY.set_index("detector").loc["YOLO11n"]
SPLIT_PER_SEED = pd.DataFrame([{
    "seed": run.seed, "map50": RESULTS[run.run_id].test["map50"],
    "far_per_1000": RESULTS[run.run_id].test["far_per_1000"], "threshold": RESULTS[run.run_id].threshold,
    "n_test": len(GT_OFFICIAL["test"].sample_ids), "n_negatives": RESULTS[run.run_id].test["n_negatives"],
} for run in OFFICIAL_RUNS])
SPLIT_INFLUENCE = pd.DataFrame([
    {"partition": "curated (deduplicated)", "n_test": len(GT["test"].sample_ids),
     "n_negatives": GT["test"].n_negatives, "n_seeds": int(_y.n_seeds),
     "map50": round(_y.map50_mean, 4), "map50_sd": round(_y.map50_std, 4),
     "far_per_1000": round(_y.far_per_1000_mean, 2), "far_sd": round(_y.far_per_1000_std, 2)},
    {"partition": "official D-Fire", "n_test": int(SPLIT_PER_SEED.n_test.iloc[0]),
     "n_negatives": int(SPLIT_PER_SEED.n_negatives.iloc[0]), "n_seeds": len(SPLIT_PER_SEED),
     "map50": round(float(SPLIT_PER_SEED.map50.mean()), 4),
     "map50_sd": round(float(SPLIT_PER_SEED.map50.std(ddof=1)), 4),
     "far_per_1000": round(float(SPLIT_PER_SEED.far_per_1000.mean()), 2),
     "far_sd": round(float(SPLIT_PER_SEED.far_per_1000.std(ddof=1)), 2)},
])
write_table(SPLIT_PER_SEED, "ext10b_split_per_seed")
write_table(SPLIT_INFLUENCE, "ext10_split_influence")
show(SPLIT_PER_SEED, "Official-partition runs")
show(SPLIT_INFLUENCE, "YOLO11n retrained on each partition")

FINDINGS["official_map50_mean"] = float(SPLIT_PER_SEED.map50.mean())
FINDINGS["official_map50_sd"] = float(SPLIT_PER_SEED.map50.std(ddof=1))
FINDINGS["official_far_mean"] = float(SPLIT_PER_SEED.far_per_1000.mean())
FINDINGS["official_far_sd"] = float(SPLIT_PER_SEED.far_per_1000.std(ddof=1))
FINDINGS["official_far_seed42"] = float(SPLIT_PER_SEED.set_index("seed").loc[42, "far_per_1000"])
FINDINGS["official_far_seed1337"] = float(SPLIT_PER_SEED.set_index("seed").loc[1337, "far_per_1000"])
FINDINGS["curated_yolo11n_map50_mean"] = float(_y.map50_mean)
FINDINGS["curated_yolo11n_map50_sd"] = float(_y.map50_std)
FINDINGS["curated_yolo11n_far_mean"] = float(_y.far_per_1000_mean)
FINDINGS["curated_yolo11n_far_sd"] = float(_y.far_per_1000_std)
FINDINGS["split_map50_change"] = FINDINGS["official_map50_mean"] - FINDINGS["curated_yolo11n_map50_mean"]
FINDINGS["split_far_drop"] = FINDINGS["curated_yolo11n_far_mean"] - FINDINGS["official_far_mean"]
FINDINGS["official_test_images"] = int(SPLIT_PER_SEED.n_test.iloc[0])
FINDINGS["official_test_negatives"] = int(SPLIT_PER_SEED.n_negatives.iloc[0])
print(f"mAP50 change {FINDINGS['split_map50_change']:+.4f} (seed SD on the curated partition "
      f"{FINDINGS['curated_yolo11n_map50_sd']:.4f}); FAR drop {FINDINGS['split_far_drop']:.2f} per 1,000")

## 13. Extended analysis: the longer version of the paper

The condensed journal version keeps eight tables and two figures. The longer version of the manuscript carried thirteen tables and seven figures. All thirteen tables are among those computed above; of the seven figures, two are Fig. 1 and Fig. 2 of the journal version and the other five are drawn here. The correspondence is:

| Longer version | Content | File |
|---|---|---|
| Table 1, Table 2 | Dataset composition; object scale by class | `ext_dataset.csv`, `ext_object_scale.csv` |
| Table 3 | The four detectors and their cost | `ext11_capacity.csv` |
| Table 4, Fig. 1 | Per-run and summarised test results | `ext01_per_seed.csv`, `ext02_summary.csv`, `Fig1.png` |
| Table 5, Fig. 2 | FAR at fixed recall; FAR against recall (seed 42) | `ext03_far_at_recall_per_seed.csv`, `ext03b_far_at_recall_mean.csv`, `ExtFig2_far_vs_recall.png` |
| Table 6 | Paired comparisons, seed-averaged and seed 42 | `ext09b_bootstrap_seed_averaged.csv`, `ext09_bootstrap_seed42.csv` |
| Table 7 | rho computed two ways | `ext06_rank_correlation_two_ways.csv` |
| Table 8, Fig. 3 | rho over the 81 single-seed studies | `ext07_rho_distribution.csv`, `ext07b_rho_81_combinations.csv`, `ExtFig3_rho_distribution.png` |
| Table 9 | Exact permutation tests at n = 4 | `ext08_exact_permutation_tests.csv` |
| Table 10 | Permutation floor, n = 3 to 10 | `ext08b_permutation_floor.csv` |
| Table 11, Fig. 4 | Predictors of the alarm ordering; FAR against accuracy and capacity | `ext12_alarm_predictors.csv`, `Fig2.png` |
| Table 12, Fig. 5 | Expected cost and its rank correlation with mAP50 | `ext04_expected_cost_per_seed.csv`, `ext05_cost_rank_correlation.csv`, `ExtFig5_expected_cost.png` |
| Fig. 6 | AP per class | `ExtFig6_ap_per_class.png` |
| Table 13, Fig. 7 | Split influence | `ext10_split_influence.csv`, `ext10b_split_per_seed.csv`, `ExtFig7_split_influence.png` |

Two early draft tables of the original project (`versi2_korelasi_peringkat.csv`, `versi2_pengaruh_split.csv`) are not reproduced separately: the first is identical in content to `ext06`, and the second predates the second official-partition seed, so its single official row is the seed-42 row of `ext10b`.

In [ ]:
# Longer version, Fig. 3: distribution of rho over the 81 single-seed studies.
fig, ax = plt.subplots(figsize=(4.8, 3.2))
_vals, _cnts = np.unique(np.round(RHO_81, 4), return_counts=True)
ax.bar(_vals, _cnts, width=0.12, color="0.65", edgecolor="black", linewidth=0.6)
ax.axvline(RHO_MEANS, color="#c44e52", linestyle="--", linewidth=1.4,
           label=f"three-seed mean ranking (ρ = {RHO_MEANS:.2f})")
ax.axvline(float(RHO_81.mean()), color="#4c72b0", linestyle=":", linewidth=1.4,
           label=f"mean over all {len(RHO_81)} draws (ρ = {RHO_81.mean():.2f})")
ax.set_xlabel("Spearman ρ between mAP50 rank and false-alarm rank")
ax.set_ylabel("Number of single-seed draws")
ax.set_title(f"All {len(RHO_81)} ways to pick one seed per detector")
ax.legend(frameon=False, fontsize=7.5, loc="upper left")
finish_figure(fig, "ExtFig3_rho_distribution")

# Longer version, Fig. 2: false alarms against recall, seed 42.
fig, ax = plt.subplots(figsize=(4.8, 3.6))
for det in SUMMARY.detector:
    run = next(r for r in CURATED_RUNS if r.detector == det and r.seed == 42)
    curve = RESULTS[run.run_id].curve.sort_values("recall")
    ax.plot(curve.recall, curve.far_per_1000, marker=MARKER[det], markersize=3.5, color=COLOUR[det],
            linewidth=1.2, label=det)
ax.set_yscale("log")
ax.set_xlabel("Recall (mean of fire and smoke)")
ax.set_ylabel("False alarms per 1,000 verified negatives\n(log scale)")
ax.set_title("Compared at equal recall (seed 42)")
ax.legend(frameon=False, fontsize=7.5, loc="upper left")
finish_figure(fig, "ExtFig2_far_vs_recall")

# Longer version, Fig. 5: expected cost against the cost ratio.
fig, ax = plt.subplots(figsize=(4.8, 3.4))
for det in SUMMARY.detector:
    sub = COST[COST.detector == det].groupby("cost_ratio").cost_test.mean()
    ax.plot(sub.index, sub.values, marker=MARKER[det], markersize=4.5, color=COLOUR[det], linewidth=1.3, label=det)
ax.set_xscale("log")
ax.set_xticks(list(COST_RATIOS))
ax.set_xticklabels([str(r) for r in COST_RATIOS])
ax.set_xlabel("Cost ratio  $C_{miss}/C_{alarm}$")
ax.set_ylabel("Expected cost at test\n(threshold selected on validation)")
ax.set_title("Lower is better · three-seed mean")
ax.legend(frameon=False, fontsize=7.5, loc="upper left")
finish_figure(fig, "ExtFig5_expected_cost")

# Longer version, Fig. 6: AP per class.
fig, ax = plt.subplots(figsize=(4.8, 3.2))
_x = np.arange(len(SUMMARY))
ax.bar(_x - 0.19, SUMMARY.ap_fire_mean, 0.36, yerr=SUMMARY.ap_fire_std, capsize=2.5, color="#c44e52",
       edgecolor="black", linewidth=0.5, label="AP fire")
ax.bar(_x + 0.19, SUMMARY.ap_smoke_mean, 0.36, yerr=SUMMARY.ap_smoke_std, capsize=2.5, color="#8172b2",
       edgecolor="black", linewidth=0.5, label="AP smoke")
ax.set_xticks(_x)
ax.set_xticklabels(SUMMARY.detector, rotation=12, ha="right", fontsize=7.5)
ax.set_ylabel("AP at IoU 0.50")
ax.set_title("Fire is the harder class in every detector")
ax.legend(frameon=False, fontsize=7.5)
finish_figure(fig, "ExtFig6_ap_per_class")

# Longer version, Fig. 7: split influence.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5.6, 2.9))
_pos = np.arange(len(SPLIT_INFLUENCE))
ax1.bar(_pos, SPLIT_INFLUENCE.map50, 0.5, yerr=SPLIT_INFLUENCE.map50_sd.fillna(0), capsize=3,
        color=["#4c72b0", "#dd8452"], edgecolor="black", linewidth=0.5)
ax1.set_xticks(_pos)
ax1.set_xticklabels(["custom\n(dedup)", "official\nD-Fire"], fontsize=7.5)
ax1.set_ylabel("mAP50")
ax1.set_ylim(0.60, 0.76)
ax1.set_title("Detection quality", fontsize=9)
ax2.bar(_pos, SPLIT_INFLUENCE.far_per_1000, 0.5, yerr=SPLIT_INFLUENCE.far_sd.fillna(0), capsize=3,
        color=["#4c72b0", "#dd8452"], edgecolor="black", linewidth=0.5)
ax2.set_xticks(_pos)
ax2.set_xticklabels(["custom\n(dedup)", "official\nD-Fire"], fontsize=7.5)
ax2.set_ylabel("False alarms / 1,000")
ax2.set_title("Alarm behaviour", fontsize=9)
fig.suptitle("YOLO11n retrained on each partition", fontsize=9.5)
finish_figure(fig, "ExtFig7_split_influence")
print("Figures written:", ", ".join(sorted(p.name for p in FIGURE_DIR.glob("*.png"))))

## 14. Automated comparison against the expected values

Every number computed above is checked in three layers.

1. **Against the original result tables.** The CSV files that the original analysis wrote, and from which the paper was typeset, are embedded below verbatim except that column names and a few descriptive labels are translated into English. They are `dataset.csv` and `ukuran_objek.csv` (Table 1 and object scale) and `v2_tabel1` to `v2_tabel12` with their `b` variants, where `v2_tabelN` corresponds to `extNN` here. Each computed table must match them cell by cell. Because this notebook reproduces the original rounding, the tolerance is 1e-6, far below the last digit stored (1e-5 for metrics, 1e-3 for rates and intervals); in practice the differences are zero.
2. **Against the paper's tables.** The values printed in Tables 1 to 8 of the submitted manuscript are written out below. A computed value passes when it rounds to the printed one, that is, when it lies within half a unit of the last printed digit. For Table 5 the comparison uses the three-decimal values of the result table, which is what the printed values were rounded from.
3. **Against the in-text numbers** of the manuscript: the ρ values and their 81-draw distribution, the exact *p*-values, the sweep-floor count, the split-influence figures, the class gap, the multiplicity calculation and the rest.

The figures are drawn directly from tables that pass the first two layers. In addition, each PNG is hashed and compared with the published file; with matplotlib 3.11 they are identical byte for byte. Because another matplotlib version may rasterise the same data differently, a hash difference is listed as information, not as a failure.

In [ ]:
# The result tables written by the original analysis, with column names and descriptive labels
# translated into English. Values are unchanged. The file each one comes from is listed above.
EXPECTED_ARTIFACTS: dict[str, str] = {
    "ext_dataset": """\
partition,images,images_with_objects,verified_negatives,percent_negative,fire_boxes,smoke_boxes,images_with_fire,images_with_smoke,median_width,median_height,manifest_hash_prefix,frozen
train,14852,7938,6914,46.6,10256,8059,4067,7139,1027,660,063f26816180,False
val,3418,1927,1491,43.6,2162,2005,849,1765,1280,720,9fe02989d64c,False
test,3257,1824,1433,44.0,2267,1790,906,1621,1024,664,6246e1bcca79,True
""",
    "ext_object_scale": """\
partition,class,n_boxes,median_area_fraction,mean_area_fraction,percent_below_1pct_area
train,fire,10256,0.00519,0.02418,65.3
train,smoke,8059,0.15273,0.23035,19.3
test,fire,2267,0.00625,0.02623,61.6
test,smoke,1790,0.15179,0.23715,17.2
""",
    "ext01_per_seed": """\
detector,seed,threshold,map50,map50_95,ap_fire,ap_smoke,precision,recall,f1,far_per_1000,n_negatives,n_negatives_alarmed
ResNet18+FPN+FCOS,42,0.45,0.66107,0.31743,0.57554,0.74659,0.64461,0.65638,0.64704,11.863,1433,17
ResNet18+FPN+FCOS,1337,0.5,0.66763,0.31814,0.5826,0.75265,0.76327,0.58754,0.6588,6.978,1433,10
ResNet18+FPN+FCOS,2024,0.5,0.66514,0.31568,0.58275,0.74754,0.75641,0.58325,0.65405,6.978,1433,10
YOLO11n,42,0.25,0.70996,0.39525,0.64831,0.77162,0.71951,0.68361,0.70098,25.82,1433,37
YOLO11n,1337,0.25,0.70723,0.39429,0.63723,0.77723,0.7112,0.6828,0.69662,33.496,1433,48
YOLO11n,2024,0.25,0.71711,0.39828,0.65294,0.78129,0.7252,0.67942,0.70148,30.705,1433,44
YOLO11s,42,0.3,0.72662,0.40288,0.65738,0.79586,0.75624,0.67433,0.71247,32.1,1433,46
YOLO11s,1337,0.2,0.70805,0.39634,0.646,0.77011,0.69009,0.70605,0.69797,31.403,1433,45
YOLO11s,2024,0.25,0.71637,0.3998,0.65411,0.77864,0.72889,0.68626,0.70693,25.82,1433,37
YOLOv8s,42,0.25,0.71818,0.40393,0.65441,0.78195,0.73698,0.70278,0.71928,24.424,1433,35
YOLOv8s,1337,0.25,0.72008,0.40624,0.65621,0.78395,0.73005,0.70324,0.71627,15.352,1433,22
YOLOv8s,2024,0.3,0.72447,0.40414,0.66794,0.78101,0.77134,0.68448,0.725,17.446,1433,25
""",
    "ext02_summary": """\
detector,map50_mean,map50_std,map50_95_mean,map50_95_std,ap_fire_mean,ap_fire_std,ap_smoke_mean,ap_smoke_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,far_per_1000_mean,far_per_1000_std,threshold_mean,threshold_std,n_seeds
ResNet18+FPN+FCOS,0.6646133333333334,0.0033115605586087346,0.31708333333333333,0.0012661095265944745,0.5802966666666667,0.004120076860124499,0.7489266666666667,0.0032592995157447032,0.7214300000000001,0.06661643340798125,0.6090566666666667,0.04103930351910632,0.6532966666666667,0.005916082600279878,8.606333333333334,2.8203560649913224,0.48333333333333334,0.028867513459481298,3
YOLO11n,0.7114333333333334,0.0051021204742080755,0.39593999999999996,0.0020825705270170773,0.64616,0.008072663748726311,0.7767133333333334,0.004855659927685853,0.7186366666666667,0.007040740964794326,0.6819433333333333,0.002222483595740004,0.6996933333333333,0.0026733000829187417,30.007,3.8853116992076715,0.25,0.0,3
YOLO11s,0.7170133333333334,0.009301700561366977,0.3996733333333333,0.003271839441863445,0.6524966666666666,0.005859030067624934,0.7815366666666667,0.013117112233008254,0.7250733333333333,0.03323974779286595,0.68888,0.016021482453256297,0.70579,0.007316911916922337,29.774333333333335,3.4422400168107594,0.25,0.04999999999999999,3
YOLOv8s,0.7209099999999999,0.0032260967127473966,0.40477,0.0012773801313626112,0.65952,0.007347264797188095,0.7823033333333335,0.0015015103507246973,0.7461233333333332,0.022111454799115696,0.6968333333333333,0.010700772557779783,0.7201833333333333,0.004434549958376051,19.074,4.750061473286423,0.26666666666666666,0.02886751345948128,3
""",
    "ext03_far_at_recall_per_seed": """\
detector,seed,far_at_recall_0.5,far_at_recall_0.6,far_at_recall_0.7,far_at_recall_0.8,highest_recall
ResNet18+FPN+FCOS,42,4.474,8.419,18.569,53.349,0.90327
ResNet18+FPN+FCOS,1337,4.454,7.44,15.812,50.068,0.90612
ResNet18+FPN+FCOS,2024,3.179,7.773,14.168,41.453,0.90594
YOLO11n,42,4.885,11.737,31.265,92.391,0.83502
YOLO11n,1337,8.219,19.349,39.329,96.504,0.84047
YOLO11n,2024,3.659,14.037,38.893,103.146,0.84331
YOLO11s,42,7.336,16.558,40.884,86.255,0.85184
YOLO11s,1337,7.331,17.082,30.238,70.626,0.83333
YOLO11s,2024,6.259,15.958,27.972,61.601,0.83821
YOLOv8s,42,7.727,13.546,23.777,59.922,0.83173
YOLOv8s,1337,5.043,8.737,15.178,43.886,0.82943
YOLOv8s,2024,6.286,10.713,20.774,51.435,0.83536
""",
    "ext03b_far_at_recall_mean": """\
detector,far_at_recall_0.5,far_at_recall_0.6,far_at_recall_0.7,far_at_recall_0.8,highest_recall
ResNet18+FPN+FCOS,4.035666666666667,7.8773333333333335,16.183,48.29,0.90511
YOLO11n,5.587666666666666,15.041000000000002,36.495666666666665,97.347,0.8396
YOLO11s,6.975333333333334,16.532666666666668,33.03133333333333,72.82733333333333,0.8411266666666667
YOLOv8s,6.352,10.998666666666665,19.909666666666666,51.74766666666667,0.8321733333333333
""",
    "ext04_expected_cost_per_seed": """\
detector,seed,cost_ratio,threshold_selected_on_val,recall_test,far_test,cost_test,at_sweep_floor
ResNet18+FPN+FCOS,42,1,0.25,0.84108,0.08793,0.24685,False
ResNet18+FPN+FCOS,42,2,0.2,0.86614,0.13678,0.40449,False
ResNet18+FPN+FCOS,42,5,0.2,0.86614,0.13678,0.80606,False
ResNet18+FPN+FCOS,42,10,0.15,0.8846,0.23238,1.38642,False
ResNet18+FPN+FCOS,42,25,0.15,0.8846,0.23238,3.11749,False
ResNet18+FPN+FCOS,42,50,0.05,0.90327,1.0,5.83665,True
ResNet18+FPN+FCOS,1337,1,0.25,0.83994,0.09211,0.25217,False
ResNet18+FPN+FCOS,1337,2,0.2,0.86567,0.15841,0.42706,False
ResNet18+FPN+FCOS,1337,5,0.2,0.86567,0.15841,0.83004,False
ResNet18+FPN+FCOS,1337,10,0.15,0.88805,0.26239,1.38188,False
ResNet18+FPN+FCOS,1337,25,0.1,0.90015,0.54431,3.04055,False
ResNet18+FPN+FCOS,1337,50,0.1,0.90015,0.54431,5.53678,False
ResNet18+FPN+FCOS,2024,1,0.25,0.84406,0.08932,0.24526,False
ResNet18+FPN+FCOS,2024,2,0.25,0.84406,0.08932,0.4012,False
ResNet18+FPN+FCOS,2024,5,0.2,0.86851,0.14515,0.80259,False
ResNet18+FPN+FCOS,2024,10,0.15,0.88996,0.26239,1.36277,False
ResNet18+FPN+FCOS,2024,25,0.1,0.90193,0.60921,3.06097,False
ResNet18+FPN+FCOS,2024,50,0.1,0.90193,0.60921,5.51273,False
YOLO11n,42,1,0.05,0.83502,0.1291,0.29408,True
YOLO11n,42,2,0.05,0.83502,0.1291,0.45906,True
YOLO11n,42,5,0.05,0.83502,0.1291,0.954,True
YOLO11n,42,10,0.05,0.83502,0.1291,1.7789,True
YOLO11n,42,25,0.05,0.83502,0.1291,4.25359,True
YOLO11n,42,50,0.05,0.83502,0.1291,8.37809,True
YOLO11n,1337,1,0.05,0.84047,0.13957,0.29909,True
YOLO11n,1337,2,0.05,0.84047,0.13957,0.45862,True
YOLO11n,1337,5,0.05,0.84047,0.13957,0.93719,True
YOLO11n,1337,10,0.05,0.84047,0.13957,1.73482,True
YOLO11n,1337,25,0.05,0.84047,0.13957,4.1277,True
YOLO11n,1337,50,0.05,0.84047,0.13957,8.11583,True
YOLO11n,2024,1,0.05,0.84331,0.14934,0.30602,True
YOLO11n,2024,2,0.05,0.84331,0.14934,0.46271,True
YOLO11n,2024,5,0.05,0.84331,0.14934,0.93278,True
YOLO11n,2024,10,0.05,0.84331,0.14934,1.71621,True
YOLO11n,2024,25,0.05,0.84331,0.14934,4.06653,True
YOLO11n,2024,50,0.05,0.84331,0.14934,7.98373,True
YOLO11s,42,1,0.1,0.80725,0.09211,0.28487,False
YOLO11s,42,2,0.05,0.85184,0.14864,0.44496,True
YOLO11s,42,5,0.05,0.85184,0.14864,0.88944,True
YOLO11s,42,10,0.05,0.85184,0.14864,1.63025,True
YOLO11s,42,25,0.05,0.85184,0.14864,3.85266,True
YOLO11s,42,50,0.05,0.85184,0.14864,7.55667,True
YOLO11s,1337,1,0.05,0.83333,0.09839,0.26507,True
YOLO11s,1337,2,0.05,0.83333,0.09839,0.43174,True
YOLO11s,1337,5,0.05,0.83333,0.09839,0.93175,True
YOLO11s,1337,10,0.05,0.83333,0.09839,1.7651,True
YOLO11s,1337,25,0.05,0.83333,0.09839,4.26517,True
YOLO11s,1337,50,0.05,0.83333,0.09839,8.43194,True
YOLO11s,2024,1,0.05,0.83821,0.10398,0.26577,True
YOLO11s,2024,2,0.05,0.83821,0.10398,0.42756,True
YOLO11s,2024,5,0.05,0.83821,0.10398,0.91293,True
YOLO11s,2024,10,0.05,0.83821,0.10398,1.72187,True
YOLO11s,2024,25,0.05,0.83821,0.10398,4.14872,True
YOLO11s,2024,50,0.05,0.83821,0.10398,8.19346,True
YOLOv8s,42,1,0.05,0.83173,0.08234,0.25062,True
YOLOv8s,42,2,0.05,0.83173,0.08234,0.41889,True
YOLOv8s,42,5,0.05,0.83173,0.08234,0.92371,True
YOLOv8s,42,10,0.05,0.83173,0.08234,1.76508,True
YOLOv8s,42,25,0.05,0.83173,0.08234,4.28918,True
YOLOv8s,42,50,0.05,0.83173,0.08234,8.49602,True
YOLOv8s,1337,1,0.05,0.82943,0.07467,0.24524,True
YOLOv8s,1337,2,0.05,0.82943,0.07467,0.4158,True
YOLOv8s,1337,5,0.05,0.82943,0.07467,0.9275,True
YOLOv8s,1337,10,0.05,0.82943,0.07467,1.78034,True
YOLOv8s,1337,25,0.05,0.82943,0.07467,4.33884,True
YOLOv8s,1337,50,0.05,0.82943,0.07467,8.60302,True
YOLOv8s,2024,1,0.05,0.83536,0.07258,0.23722,True
YOLOv8s,2024,2,0.05,0.83536,0.07258,0.40186,True
YOLOv8s,2024,5,0.05,0.83536,0.07258,0.89578,True
YOLOv8s,2024,10,0.05,0.83536,0.07258,1.71899,True
YOLOv8s,2024,25,0.05,0.83536,0.07258,4.18862,True
YOLOv8s,2024,50,0.05,0.83536,0.07258,8.30466,True
""",
    "ext05_cost_rank_correlation": """\
cost_ratio,rho_map50_vs_cost,map50_order,cost_order_cheapest_first,cheapest,best_map50,detectors_at_sweep_floor
1,0.4,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,YOLOv8s > ResNet18+FPN+FCOS > YOLO11s > YOLO11n,YOLOv8s,YOLOv8s,YOLO11n; YOLO11s; YOLOv8s
2,-0.2,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLOv8s > YOLO11s > YOLO11n,ResNet18+FPN+FCOS,YOLOv8s,YOLO11n; YOLO11s; YOLOv8s
5,-0.4,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLO11s > YOLOv8s > YOLO11n,ResNet18+FPN+FCOS,YOLOv8s,YOLO11n; YOLO11s; YOLOv8s
10,-0.8,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLO11s > YOLO11n > YOLOv8s,ResNet18+FPN+FCOS,YOLOv8s,YOLO11n; YOLO11s; YOLOv8s
25,-0.8,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLO11s > YOLO11n > YOLOv8s,ResNet18+FPN+FCOS,YOLOv8s,YOLO11n; YOLO11s; YOLOv8s
50,-0.8,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLO11s > YOLO11n > YOLOv8s,ResNet18+FPN+FCOS,YOLOv8s,ResNet18+FPN+FCOS; YOLO11n; YOLO11s; YOLOv8s
""",
    "ext06_rank_correlation_two_ways": """\
method,rho,map50_order,far_order_safest_first
single seed (42 only),-0.8,YOLO11s > YOLOv8s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLOv8s > YOLO11n > YOLO11s
three-seed means,-0.2,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLOv8s > YOLO11s > YOLO11n
""",
    "ext07_rho_distribution": """\
n_combinations,rho_mean,rho_sd,rho_min,rho_q1,rho_median,rho_q3,rho_max,share_rho_le_-0.5,share_rho_le_-0.8,share_rho_ge_0,rho_three_seed_means
81,-0.4,0.2324,-0.8,-0.4,-0.4,-0.2,-0.2,0.2222,0.2222,0.0,-0.2
""",
    "ext07b_rho_81_combinations": """\
seed_ResNet18+FPN+FCOS,seed_YOLO11n,seed_YOLO11s,seed_YOLOv8s,rho
42,42,42,42,-0.8
42,42,42,1337,-0.8
42,42,42,2024,-0.8
42,42,1337,42,-0.2
42,42,1337,1337,-0.2
42,42,1337,2024,-0.2
42,42,2024,42,-0.4
42,42,2024,1337,-0.4
42,42,2024,2024,-0.4
42,1337,42,42,-0.4
42,1337,42,1337,-0.4
42,1337,42,2024,-0.4
42,1337,1337,42,-0.2
42,1337,1337,1337,-0.2
42,1337,1337,2024,-0.2
42,1337,2024,42,-0.2
42,1337,2024,1337,-0.2
42,1337,2024,2024,-0.2
42,2024,42,42,-0.8
42,2024,42,1337,-0.8
42,2024,42,2024,-0.8
42,2024,1337,42,-0.2
42,2024,1337,1337,-0.2
42,2024,1337,2024,-0.2
42,2024,2024,42,-0.4
42,2024,2024,1337,-0.4
42,2024,2024,2024,-0.4
1337,42,42,42,-0.8
1337,42,42,1337,-0.8
1337,42,42,2024,-0.8
1337,42,1337,42,-0.2
1337,42,1337,1337,-0.2
1337,42,1337,2024,-0.2
1337,42,2024,42,-0.4
1337,42,2024,1337,-0.4
1337,42,2024,2024,-0.4
1337,1337,42,42,-0.4
1337,1337,42,1337,-0.4
1337,1337,42,2024,-0.4
1337,1337,1337,42,-0.2
1337,1337,1337,1337,-0.2
1337,1337,1337,2024,-0.2
1337,1337,2024,42,-0.2
1337,1337,2024,1337,-0.2
1337,1337,2024,2024,-0.2
1337,2024,42,42,-0.8
1337,2024,42,1337,-0.8
1337,2024,42,2024,-0.8
1337,2024,1337,42,-0.2
1337,2024,1337,1337,-0.2
1337,2024,1337,2024,-0.2
1337,2024,2024,42,-0.4
1337,2024,2024,1337,-0.4
1337,2024,2024,2024,-0.4
2024,42,42,42,-0.8
2024,42,42,1337,-0.8
2024,42,42,2024,-0.8
2024,42,1337,42,-0.2
2024,42,1337,1337,-0.2
2024,42,1337,2024,-0.2
2024,42,2024,42,-0.4
2024,42,2024,1337,-0.4
2024,42,2024,2024,-0.4
2024,1337,42,42,-0.4
2024,1337,42,1337,-0.4
2024,1337,42,2024,-0.4
2024,1337,1337,42,-0.2
2024,1337,1337,1337,-0.2
2024,1337,1337,2024,-0.2
2024,1337,2024,42,-0.2
2024,1337,2024,1337,-0.2
2024,1337,2024,2024,-0.2
2024,2024,42,42,-0.8
2024,2024,42,1337,-0.8
2024,2024,42,2024,-0.8
2024,2024,1337,42,-0.2
2024,2024,1337,1337,-0.2
2024,2024,1337,2024,-0.2
2024,2024,2024,42,-0.4
2024,2024,2024,1337,-0.4
2024,2024,2024,2024,-0.4
""",
    "ext08_exact_permutation_tests": """\
case,rho,p_two_sided_exact,n_permutations,significant_at_0.05
single seed (42 only),-0.8,0.3333,24,False
three-seed means,-0.2,0.9167,24,False
most extreme rho possible,-1.0,0.0833,24,False
""",
    "ext08b_permutation_floor": """\
n_detectors,n_permutations,most_extreme_abs_rho,smallest_two_sided_p,can_reach_0.05
3,6,1.0,0.33333,False
4,24,1.0,0.08333,False
5,120,1.0,0.01667,True
6,720,1.0,0.00278,True
7,5040,1.0,0.0004,True
8,40320,1.0,5e-05,True
9,362880,1.0,1e-05,True
10,3628800,1.0,0.0,True
""",
    "ext09_bootstrap_seed42": """\
kind,a,b,value,lower,upper,p,n
bootstrap interval of FAR,ResNet18+FPN+FCOS,,11.863,6.978,17.446,,1433
bootstrap interval of FAR,YOLO11n,,25.82,17.446,34.194,,1433
bootstrap interval of FAR,YOLO11s,,32.1,23.029,41.172,,1433
bootstrap interval of FAR,YOLOv8s,,24.424,16.748,32.816,,1433
paired test of FAR difference,ResNet18+FPN+FCOS,YOLO11n,-13.957,-23.029,-5.565,0.003,1433
paired test of FAR difference,ResNet18+FPN+FCOS,YOLO11s,-20.237,-30.705,-10.468,0.0,1433
paired test of FAR difference,ResNet18+FPN+FCOS,YOLOv8s,-12.561,-20.935,-4.885,0.002,1433
paired test of FAR difference,YOLO11n,YOLO11s,-6.281,-16.05,3.507,0.24,1433
paired test of FAR difference,YOLO11n,YOLOv8s,1.396,-7.676,9.77,0.833,1433
paired test of FAR difference,YOLO11s,YOLOv8s,7.676,-2.791,18.144,0.163,1433
""",
    "ext09b_bootstrap_seed_averaged": """\
kind,a,b,value,lower,upper,p,n,n_seeds
bootstrap interval of FAR (seed-averaged),ResNet18+FPN+FCOS,,8.607,4.885,13.026,,1433,3
bootstrap interval of FAR (seed-averaged),YOLO11n,,30.007,23.726,36.753,,1433,3
bootstrap interval of FAR (seed-averaged),YOLO11s,,29.774,23.029,36.753,,1433,3
bootstrap interval of FAR (seed-averaged),YOLOv8s,,19.074,13.724,24.895,,1433,3
paired test of FAR difference (seed-averaged),ResNet18+FPN+FCOS,YOLO11n,-21.4,-28.611,-14.887,0.0,1433,3
paired test of FAR difference (seed-averaged),ResNet18+FPN+FCOS,YOLO11s,-21.168,-28.146,-14.422,0.0,1433,3
paired test of FAR difference (seed-averaged),ResNet18+FPN+FCOS,YOLOv8s,-10.468,-16.515,-5.117,0.0,1433,3
paired test of FAR difference (seed-averaged),YOLO11n,YOLO11s,0.233,-5.583,6.281,0.957,1433,3
paired test of FAR difference (seed-averaged),YOLO11n,YOLOv8s,10.933,5.583,16.515,0.0,1433,3
paired test of FAR difference (seed-averaged),YOLO11s,YOLOv8s,10.7,5.35,16.283,0.0,1433,3
""",
    "ext10_split_influence": """\
partition,n_test,n_negatives,n_seeds,map50,map50_sd,far_per_1000,far_sd
curated (deduplicated),3257,1433,3,0.7114,0.0051,30.01,3.89
official D-Fire,4306,2005,2,0.7069,0.0021,20.45,2.82
""",
    "ext10b_split_per_seed": """\
seed,map50,far_per_1000,threshold,n_test,n_negatives
42,0.70837,18.454,0.25,4306,2005
1337,0.70545,22.444,0.25,4306,2005
""",
    "ext11_capacity": """\
detector,params_m,gflops,size_mb,latency_ms,fps,peak_vram_gb,source,family
ResNet18+FPN+FCOS,12.1703,,46.468,5.8568,170.742,0.095,efficiency harness (E0b seed 42),custom single-stage
YOLO11n,2.5902,3.22,5.45,4.6868,213.367,0.0674,YOLO efficiency benchmark (idle GPU),YOLO
YOLO11s,9.4286,10.773,19.156,5.0035,199.861,0.1083,YOLO efficiency benchmark (idle GPU),YOLO
YOLOv8s,11.1364,14.322,22.497,4.9191,203.288,0.111,YOLO efficiency benchmark (idle GPU),YOLO
""",
    "ext12_alarm_predictors": """\
predictor,rho_all_four,rho_yolo_only,predictor_order_best_first,alarm_order_safest_first
mAP50,-0.2,1.0,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLOv8s > YOLO11s > YOLO11n
mAP50-95,-0.2,1.0,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLOv8s > YOLO11s > YOLO11n
recall at operating point,-0.2,1.0,YOLOv8s > YOLO11s > YOLO11n > ResNet18+FPN+FCOS,ResNet18+FPN+FCOS > YOLOv8s > YOLO11s > YOLO11n
precision at operating point,0.4,1.0,YOLOv8s > YOLO11s > ResNet18+FPN+FCOS > YOLO11n,ResNet18+FPN+FCOS > YOLOv8s > YOLO11s > YOLO11n
highest recall attained,0.4,-0.5,ResNet18+FPN+FCOS > YOLO11s > YOLO11n > YOLOv8s,ResNet18+FPN+FCOS > YOLOv8s > YOLO11s > YOLO11n
parameter count,1.0,1.0,ResNet18+FPN+FCOS > YOLOv8s > YOLO11s > YOLO11n,ResNet18+FPN+FCOS > YOLOv8s > YOLO11s > YOLO11n
""",
}

In [ ]:
ARTIFACT_KEYS = {
    "ext_dataset": ["partition"],
    "ext_object_scale": ["partition", "class"],
    "ext01_per_seed": ["detector", "seed"],
    "ext02_summary": ["detector"],
    "ext03_far_at_recall_per_seed": ["detector", "seed"],
    "ext03b_far_at_recall_mean": ["detector"],
    "ext04_expected_cost_per_seed": ["detector", "seed", "cost_ratio"],
    "ext05_cost_rank_correlation": ["cost_ratio"],
    "ext06_rank_correlation_two_ways": ["method"],
    "ext07_rho_distribution": ["n_combinations"],
    "ext07b_rho_81_combinations": [f"seed_{d}" for d in DETECTORS],
    "ext08_exact_permutation_tests": ["case"],
    "ext08b_permutation_floor": ["n_detectors"],
    "ext09_bootstrap_seed42": ["kind", "a", "b"],
    "ext09b_bootstrap_seed_averaged": ["kind", "a", "b"],
    "ext10_split_influence": ["partition"],
    "ext10b_split_per_seed": ["seed"],
    "ext11_capacity": ["detector"],
    "ext12_alarm_predictors": ["predictor"],
}
COMPUTED_ARTIFACTS = {
    "ext_dataset": EXT_DATASET, "ext_object_scale": EXT_OBJECT_SCALE,
    "ext01_per_seed": PER_SEED, "ext02_summary": SUMMARY,
    "ext03_far_at_recall_per_seed": FAR_AT_RECALL, "ext03b_far_at_recall_mean": FAR_AT_RECALL_MEAN,
    "ext04_expected_cost_per_seed": COST, "ext05_cost_rank_correlation": COST_CORRELATION,
    "ext06_rank_correlation_two_ways": RANK_TWO_WAYS, "ext07_rho_distribution": RHO_DISTRIBUTION,
    "ext07b_rho_81_combinations": RHO_COMBINATIONS, "ext08_exact_permutation_tests": PERMUTATION_TESTS,
    "ext08b_permutation_floor": PERMUTATION_FLOOR, "ext09_bootstrap_seed42": BOOT_SEED42,
    "ext09b_bootstrap_seed_averaged": BOOT_SEED_AVG, "ext10_split_influence": SPLIT_INFLUENCE,
    "ext10b_split_per_seed": SPLIT_PER_SEED, "ext11_capacity": CAPACITY,
    "ext12_alarm_predictors": PREDICTORS,
}
ARTIFACT_TOLERANCE = 1e-6


def _as_key(value: Any) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))
    return str(value)


def _as_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    return str(value)


def compare_with_artifact(name: str, computed: pd.DataFrame, expected_csv: str, keys: list[str]) -> dict[str, Any]:
    expected = pd.read_csv(io.StringIO(expected_csv))
    missing = [c for c in expected.columns if c not in computed.columns]
    if missing:
        return {"check": name, "status": "FAIL", "detail": f"missing columns: {missing}"}
    comp = computed[list(expected.columns)].copy()
    exp = expected.copy()
    for frame in (comp, exp):
        for k in keys:
            frame[k] = frame[k].map(_as_key)
    merged = exp.merge(comp, on=keys, how="outer", suffixes=("__expected", "__computed"), indicator=True)
    unmatched = int((merged["_merge"] != "both").sum())
    worst, bad_cells = 0.0, []
    for col in expected.columns:
        if col in keys:
            continue
        e, c = merged[f"{col}__expected"], merged[f"{col}__computed"]
        if pd.api.types.is_numeric_dtype(expected[col]):
            ev = pd.to_numeric(e, errors="coerce").astype(float)
            cv = pd.to_numeric(c.map(lambda v: np.nan if v is None else v), errors="coerce").astype(float)
            diff = (ev - cv).abs()
            bad = (ev.isna() != cv.isna()) | (diff > ARTIFACT_TOLERANCE)
            if diff.notna().any():
                worst = max(worst, float(diff.max()))
        else:
            bad = e.map(_as_text) != c.map(_as_text)
        if bad.any():
            bad_cells.append(f"{col} ({int(bad.sum())} rows)")
    ok = unmatched == 0 and not bad_cells and len(comp) == len(exp)
    detail = (f"{len(exp)} rows x {len(expected.columns)} columns, max abs difference {worst:.2g}" if ok
              else f"unmatched rows {unmatched}; mismatching columns: {', '.join(bad_cells) or 'none'}")
    return {"check": name, "status": "PASS" if ok else "FAIL", "detail": detail}


ARTIFACT_CHECKS = pd.DataFrame([
    compare_with_artifact(name, COMPUTED_ARTIFACTS[name], csv_text, ARTIFACT_KEYS[name])
    for name, csv_text in EXPECTED_ARTIFACTS.items()
])
show(ARTIFACT_CHECKS, "Layer 1: computed tables against the original result tables")

In [ ]:
# Values printed in the submitted manuscript (SIVP_Ranking-Fire-Detectors/main.tex).
PAPER_TABLE1 = {   # images, with objects, verified negatives, fire boxes, smoke boxes
    "train": ("14,852", "7,938", "6,914", "10,256", "8,059"),
    "val": ("3,418", "1,927", "1,491", "2,162", "2,005"),
    "test": ("3,257", "1,824", "1,433", "2,267", "1,790"),
}
PAPER_TABLE2 = {   # params (M), GFLOPs, size (MB), latency (ms), FPS
    "ResNet18+FPN+FCOS": ("12.170", "---", "46.5", "5.86", "170.7"),
    "YOLO11n": ("2.590", "3.22", "5.4", "4.69", "213.4"),
    "YOLO11s": ("9.429", "10.77", "19.2", "5.00", "199.9"),
    "YOLOv8s": ("11.136", "14.32", "22.5", "4.92", "203.3"),
}
PAPER_TABLE3_THRESHOLD = {"ResNet18+FPN+FCOS": "0.45--0.50", "YOLO11n": "0.25", "YOLO11s": "0.20--0.30",
                          "YOLOv8s": "0.25--0.30"}
PAPER_TABLE3 = {   # mean, SD per detector in the order of DETECTORS
    "map50": (("0.6646", "0.0033"), ("0.7114", "0.0051"), ("0.7170", "0.0093"), ("0.7209", "0.0032")),
    "map50_95": (("0.3171", "0.0013"), ("0.3959", "0.0021"), ("0.3997", "0.0033"), ("0.4048", "0.0013")),
    "ap_fire": (("0.5803", "0.0041"), ("0.6462", "0.0081"), ("0.6525", "0.0059"), ("0.6595", "0.0073")),
    "ap_smoke": (("0.7489", "0.0033"), ("0.7767", "0.0049"), ("0.7815", "0.0131"), ("0.7823", "0.0015")),
    "precision": (("0.7214", "0.0666"), ("0.7186", "0.0070"), ("0.7251", "0.0332"), ("0.7461", "0.0221")),
    "recall": (("0.6091", "0.0410"), ("0.6819", "0.0022"), ("0.6889", "0.0160"), ("0.6968", "0.0107")),
    "far_per_1000": (("8.61", "2.82"), ("30.01", "3.89"), ("29.77", "3.44"), ("19.07", "4.75")),
}
PAPER_TABLE4 = {   # FAR at R = 0.5, 0.6, 0.7, 0.8 with the multiplier, then highest recall attained
    "ResNet18+FPN+FCOS": ((("4.04", None), ("7.88", None), ("16.18", None), ("48.29", None)), "0.9051"),
    "YOLO11n": ((("5.59", "1.38"), ("15.04", "1.91"), ("36.50", "2.26"), ("97.35", "2.02")), "0.8396"),
    "YOLO11s": ((("6.98", "1.73"), ("16.53", "2.10"), ("33.03", "2.04"), ("72.83", "1.51")), "0.8411"),
    "YOLOv8s": ((("6.35", "1.57"), ("11.00", "1.40"), ("19.91", "1.23"), ("51.75", "1.07")), "0.8322"),
}
PAPER_TABLE5_FAR = {   # seed-averaged FAR, interval, seed-42 FAR
    "ResNet18+FPN+FCOS": ("8.61", "4.89", "13.03", "11.86"),
    "YOLO11n": ("30.01", "23.73", "36.75", "25.82"),
    "YOLO11s": ("29.77", "23.03", "36.75", "32.10"),
    "YOLOv8s": ("19.07", "13.72", "24.90", "24.42"),
}
PAPER_TABLE5_PAIRS = {   # difference, interval, p; seed-42 difference and p
    ("ResNet18+FPN+FCOS", "YOLO11n"): ("-21.40", "-28.61", "-14.89", "< 0.001", "-13.96", "0.003"),
    ("ResNet18+FPN+FCOS", "YOLO11s"): ("-21.17", "-28.15", "-14.42", "< 0.001", "-20.24", "< 0.001"),
    ("ResNet18+FPN+FCOS", "YOLOv8s"): ("-10.47", "-16.52", "-5.12", "< 0.001", "-12.56", "0.002"),
    ("YOLO11n", "YOLO11s"): ("0.23", "-5.58", "6.28", "0.957", "-6.28", "0.240"),
    ("YOLO11n", "YOLOv8s"): ("10.93", "5.58", "16.52", "< 0.001", "1.40", "0.833"),
    ("YOLO11s", "YOLOv8s"): ("10.70", "5.35", "16.28", "< 0.001", "7.68", "0.163"),
}
PAPER_TABLE6 = {3: ("6", "0.333", "no"), 4: ("24", "0.083", "no"), 5: ("120", "0.017", "yes"),
                6: ("720", "0.0028", "yes"), 8: ("40,320", "0.00005", "yes")}
PAPER_TABLE7 = {"mAP50": ("-0.20", "+1.00"), "mAP50-95": ("-0.20", "+1.00"),
                "recall at operating point": ("-0.20", "+1.00"), "precision at operating point": ("+0.40", "+1.00"),
                "highest recall attained": ("+0.40", "-0.50"), "parameter count": ("+1.00", "+1.00")}
PAPER_TABLE8 = {   # mean cost per detector, cheapest, rho, sweep-floor column
    1: ("0.248", "0.300", "0.272", "0.244", "YOLOv8s", "+0.40", "3 YOLO"),
    2: ("0.411", "0.460", "0.435", "0.412", "ResNet", "-0.20", "3 YOLO"),
    5: ("0.813", "0.941", "0.911", "0.916", "ResNet", "-0.40", "3 YOLO"),
    10: ("1.377", "1.743", "1.706", "1.755", "ResNet", "-0.80", "3 YOLO"),
    25: ("3.073", "4.149", "4.089", "4.272", "ResNet", "-0.80", "3 YOLO"),
    50: ("5.629", "8.159", "8.061", "8.468", "ResNet", "-0.80", "all 4"),
}

# In-text numbers: (key in FINDINGS, value as printed, where it appears in the manuscript).
PAPER_INTEXT = [
    ("total_images", "21,527", "Sec. 3.1, D-Fire size"),
    ("total_verified_negatives", "9,838", "Sec. 3.1, empty annotation files"),
    ("total_fire_boxes", "14,685", "Sec. 3.1, 14,692 published minus 7 degenerate"),
    ("total_smoke_boxes", "11,854", "Sec. 3.1, 11,865 published minus 11 degenerate"),
    ("median_fire_area_test", "0.00625", "Sec. 3.1"),
    ("median_smoke_area_test", "0.15179", "Sec. 3.1"),
    ("pct_fire_boxes_below_1pct", "61.6", "Sec. 3.1"),
    ("pct_smoke_boxes_below_1pct", "17.2", "Sec. 3.1"),
    ("verified_negatives_test", "1,433", "Abstract; Sec. 3.5"),
    ("custom_latency_ms_seed42", "5.86", "Sec. 3.2"),
    ("fastest_latency_ms", "4.69", "Sec. 3.2"),
    ("custom_slower_than_fastest_pct", "25", "Sec. 3.2, 'about 25 %'"),
    ("custom_latency_ms_min_over_seeds", "5.86", "Sec. 3.2, three runs span"),
    ("custom_latency_ms_max_over_seeds", "7.77", "Sec. 3.2, three runs span"),
    ("custom_fps_min_over_seeds", "128.7", "Sec. 3.2, three runs span"),
    ("custom_fps_max_over_seeds", "170.7", "Sec. 3.2, three runs span"),
    ("official_partition_sizes", "14496 / 2725 / 4306", "Sec. 3.5"),
    ("official_test_images_in_curated_train", "2,990", "Sec. 3.5"),
    ("official_test_images", "4,306", "Sec. 4.7"),
    ("official_test_negatives", "2,005", "Sec. 3.5; Sec. 4.7"),
    ("yolo_map50_min", "0.711", "Sec. 4.1"),
    ("yolo_map50_max", "0.721", "Sec. 4.1"),
    ("custom_map50_gap_min_points", "4.7", "Sec. 4.1"),
    ("custom_map50_gap_max_points", "5.6", "Sec. 4.1"),
    ("safest_detector_far", "8.61", "Abstract; Sec. 4.1"),
    ("other_detectors_far_min", "19.07", "Abstract; Sec. 4.1"),
    ("other_detectors_far_max", "30.01", "Abstract; Sec. 4.1"),
    ("custom_recall_mean", "0.609", "Abstract; Conclusion"),
    ("most_accurate_recall_mean", "0.697", "Abstract; Conclusion"),
    ("runs_with_fire_ap_below_smoke", "12", "Sec. 4.1, 'in every case'"),
    ("fire_smoke_gap_detector_means_min", "0.123", "Sec. 4.1"),
    ("fire_smoke_gap_detector_means_max", "0.169", "Sec. 4.1"),
    ("alarm_gap_fold_at_operating_point", "3.5", "Sec. 1; Sec. 4.2"),
    ("matched_recall_ratio_min", "1.1", "Abstract; Sec. 4.2"),
    ("matched_recall_ratio_max", "2.3", "Abstract; Sec. 4.2"),
    ("custom_highest_recall", "0.9051", "Sec. 4.2"),
    ("yolo_highest_recall_min", "0.832", "Sec. 4.2"),
    ("yolo_highest_recall_max", "0.841", "Sec. 4.2"),
    ("separable_pairs_seed_averaged", "5", "Abstract; Sec. 4.3"),
    ("separable_pairs_seed42", "3", "Sec. 4.3"),
    ("rho_seed42", "-0.80", "Sec. 4.4"),
    ("rho_three_seed_means", "-0.20", "Abstract; Sec. 4.4"),
    ("draws_with_rho_-0.80", "18", "Sec. 4.4"),
    ("draws_with_rho_-0.40", "27", "Sec. 4.4"),
    ("draws_with_rho_-0.20", "36", "Sec. 4.4"),
    ("rho_81_mean", "-0.40", "Sec. 4.4"),
    ("rho_81_sd", "0.23", "Sec. 4.4"),
    ("rho_81_draws_non_negative", "0", "Sec. 4.4"),
    ("rho_81_share_strongest_pct", "22.2", "Sec. 4.4; Conclusion"),
    ("rho_81_magnitude_ratio", "4", "Sec. 4.4, 'four-fold'"),
    ("exact_p_seed42", "0.333", "Sec. 4.4"),
    ("exact_p_three_seed_means", "0.917", "Sec. 4.4"),
    ("exact_p_floor_n4", "0.083", "Abstract; Sec. 4.4"),
    ("exact_p_floor_n3", "0.333", "Sec. 4.5"),
    ("n_predictors_screened", "6", "Sec. 4.5"),
    ("multiplicity_any_direction", "0.41", "Sec. 4.5"),
    ("multiplicity_correct_direction", "0.23", "Sec. 4.5"),
    ("yolo_cells_at_sweep_floor", "53", "Sec. 4.6"),
    ("yolo_cells_total", "54", "Sec. 4.6"),
    ("yolo_cells_off_floor", "YOLO11s seed 42 r=1", "Sec. 4.6, 'one YOLO11s seed at r = 1'"),
    ("custom_cells_at_sweep_floor", "1", "Sec. 4.6"),
    ("custom_cells_at_floor_where", "seed 42 r=50", "Sec. 4.6, 'one seed at r = 50'"),
    ("official_map50_mean", "0.7069", "Sec. 4.7"),
    ("official_map50_sd", "0.0021", "Sec. 4.7"),
    ("curated_yolo11n_map50_mean", "0.7114", "Sec. 4.7"),
    ("curated_yolo11n_map50_sd", "0.0051", "Sec. 4.7"),
    ("split_map50_change", "-0.0045", "Sec. 4.7"),
    ("official_far_mean", "20.45", "Sec. 4.7"),
    ("official_far_sd", "2.82", "Sec. 4.7"),
    ("curated_yolo11n_far_mean", "30.01", "Sec. 4.7"),
    ("curated_yolo11n_far_sd", "3.89", "Sec. 4.7"),
    ("official_far_seed42", "18.45", "Sec. 4.7"),
    ("official_far_seed1337", "22.44", "Sec. 4.7"),
    ("split_far_drop", "9.6", "Sec. 4.7"),
    ("map50_seed_sd_min", "0.003", "Sec. 5, noise floor"),
    ("map50_seed_sd_max", "0.009", "Sec. 5, noise floor"),
    ("far_seed_sd_min", "2.8", "Sec. 5, noise floor"),
    ("far_seed_sd_max", "4.8", "Sec. 5, noise floor"),
]

In [ ]:
def _normalise_text(text: str) -> str:
    return text.replace("--", "-").replace("–", "-").replace("−", "-").strip()


def paper_check(layer: str, item: str, computed: Any, printed: str, note: str = "") -> dict[str, Any]:
    """Pass when the computed value rounds to the printed one (within half a unit of its last digit)."""
    text = _normalise_text(printed)
    if text.startswith("<"):
        bound = float(text[1:].strip())
        ok = computed is not None and float(computed) < bound
        return {"layer": layer, "item": item, "computed": computed, "printed": printed,
                "tolerance": f"< {bound}", "status": "PASS" if ok else "FAIL", "note": note}
    number = text.replace(",", "").replace("+", "")
    try:
        expected = float(number)
    except ValueError:
        ok = _normalise_text(str(computed)) == text
        return {"layer": layer, "item": item, "computed": computed, "printed": printed,
                "tolerance": "exact text", "status": "PASS" if ok else "FAIL", "note": note}
    decimals = len(number.split(".")[1]) if "." in number else 0
    tolerance = 0.5 * 10 ** (-decimals) + 1e-9
    ok = computed is not None and not pd.isna(computed) and abs(float(computed) - expected) <= tolerance
    return {"layer": layer, "item": item, "computed": computed, "printed": printed,
            "tolerance": round(tolerance, decimals + 2), "status": "PASS" if ok else "FAIL", "note": note}


_checks: list[dict[str, Any]] = []
_ds = EXT_DATASET.set_index("partition")
for part, printed in PAPER_TABLE1.items():
    for col, text in zip(("images", "images_with_objects", "verified_negatives", "fire_boxes", "smoke_boxes"), printed):
        _checks.append(paper_check("Table 1", f"{part} {col}", int(_ds.loc[part, col]), text))

_cap = CAPACITY.set_index("detector")
for det, printed in PAPER_TABLE2.items():
    for col, text in zip(("params_m", "gflops", "size_mb", "latency_ms", "fps"), printed):
        value = _cap.loc[det, col]
        if text == "---":
            _checks.append(paper_check("Table 2", f"{det} {col}", "---" if pd.isna(value) else value, text,
                                       "recorded GPU value"))
        else:
            _checks.append(paper_check("Table 2", f"{det} {col}", float(value), text, "recorded GPU value"))

_sm = SUMMARY.set_index("detector")
for det in DETECTORS:
    _checks.append(paper_check("Table 3", f"{det} threshold range", threshold_range(det), PAPER_TABLE3_THRESHOLD[det]))
for col, per_det in PAPER_TABLE3.items():
    for det, (mean_text, sd_text) in zip(DETECTORS, per_det):
        _checks.append(paper_check("Table 3", f"{det} {col} mean", float(_sm.loc[det, f"{col}_mean"]), mean_text))
        _checks.append(paper_check("Table 3", f"{det} {col} SD", float(_sm.loc[det, f"{col}_std"]), sd_text))

_fr = FAR_AT_RECALL_MEAN.set_index("detector")
for det, (cells, highest) in PAPER_TABLE4.items():
    for target, (far_text, ratio_text) in zip(FIXED_RECALLS, cells):
        value = float(_fr.loc[det, f"far_at_recall_{target}"])
        _checks.append(paper_check("Table 4", f"{det} FAR at R={target}", value, far_text))
        if ratio_text is not None:
            ratio = value / float(_fr.loc["ResNet18+FPN+FCOS", f"far_at_recall_{target}"])
            _checks.append(paper_check("Table 4", f"{det} multiplier at R={target}", ratio, ratio_text))
    _checks.append(paper_check("Table 4", f"{det} highest recall", float(_fr.loc[det, "highest_recall"]), highest))

_ba = BOOT_SEED_AVG.set_index(["a", "b"])
_b42 = BOOT_SEED42.set_index(["a", "b"])
for det, (value, lower, upper, s42) in PAPER_TABLE5_FAR.items():
    row = _ba.loc[(det, "")]
    for name, comp, text in (("FAR", row.value, value), ("lower", row.lower, lower), ("upper", row.upper, upper),
                             ("seed-42 FAR", _b42.loc[(det, ""), "value"], s42)):
        _checks.append(paper_check("Table 5", f"{det} {name}", float(comp), text, "three-decimal table value"))
for (a, b), (diff, lower, upper, p, d42, p42) in PAPER_TABLE5_PAIRS.items():
    row, row42 = _ba.loc[(a, b)], _b42.loc[(a, b)]
    pair = f"{SHORT_NAME[a]} - {SHORT_NAME[b]}"
    for name, comp, text in (("difference", row.value, diff), ("lower", row.lower, lower),
                             ("upper", row.upper, upper), ("p", row.p, p),
                             ("seed-42 difference", row42.value, d42), ("seed-42 p", row42.p, p42)):
        _checks.append(paper_check("Table 5", f"{pair} {name}", float(comp), text, "three-decimal table value"))

_pf = PERMUTATION_FLOOR.set_index("n_detectors")
for n, (pairings, p_text, reach) in PAPER_TABLE6.items():
    _checks.append(paper_check("Table 6", f"n={n} pairings", int(_pf.loc[n, "n_permutations"]), pairings))
    _checks.append(paper_check("Table 6", f"n={n} smallest p", float(_pf.loc[n, "smallest_two_sided_p"]), p_text))
    _checks.append(paper_check("Table 6", f"n={n} can reach 0.05", "yes" if _pf.loc[n, "can_reach_0.05"] else "no", reach))

_pr = PREDICTORS.set_index("predictor")
for name, (all_four, yolo_only) in PAPER_TABLE7.items():
    _checks.append(paper_check("Table 7", f"{name} (all four)", float(_pr.loc[name, "rho_all_four"]), all_four))
    _checks.append(paper_check("Table 7", f"{name} (YOLO only)", float(_pr.loc[name, "rho_yolo_only"]), yolo_only))

_t8 = TABLE8.set_index("r")
_cc = COST_CORRELATION.set_index("cost_ratio")
for ratio, printed in PAPER_TABLE8.items():
    mean_cost = COST[COST.cost_ratio == ratio].groupby("detector").cost_test.mean()
    for det, text in zip(DETECTORS, printed[:4]):
        _checks.append(paper_check("Table 8", f"r={ratio} {det} cost", float(mean_cost[det]), text))
    _checks.append(paper_check("Table 8", f"r={ratio} cheapest", _t8.loc[ratio, "cheapest"], printed[4]))
    _checks.append(paper_check("Table 8", f"r={ratio} rho", float(_cc.loc[ratio, "rho_map50_vs_cost"]), printed[5]))
    _checks.append(paper_check("Table 8", f"r={ratio} at sweep floor", _t8.loc[ratio, "at sweep floor"], printed[6]))

for key, printed, where in PAPER_INTEXT:
    _checks.append(paper_check("In-text", key, FINDINGS.get(key), printed, where))

for _, row in REPRODUCTION_CHECK.iterrows():
    _checks.append({"layer": "Reproduction check", "item": row.quantity, "computed": row.recomputed,
                    "printed": str(row.reference), "tolerance": row.tolerance, "status": row.status,
                    "note": "Sec. 3.5 of the paper"})

# SHA-256 of the published figure files. With matplotlib 3.11 the regenerated PNGs are identical
# byte for byte; another matplotlib version may render the same data with different pixels, so a
# hash difference is reported for information only and is not a failure.
PUBLISHED_FIGURE_SHA256 = {
    "Fig1": "3b0c7352158f0453842c85d3c81c20886473eb1cd11839224880b16562d09a18",
    "Fig2": "407e96e0747141b1c124860f3484d7e8fd35fe43136bd4849c2908390baf507a",
    "ExtFig2_far_vs_recall": "740c1add85d2b0a7f8b455709049161963345b95cd26968c383e3e286c322437",
    "ExtFig3_rho_distribution": "37f1ef232035336cd15821db23c28cbab08bcd57070b8cf6921f95366f7b58c4",
    "ExtFig5_expected_cost": "62d0d22657fcdbd72227777cda760fb2fe9b389e51f422cdf0ec960c03cd4cb8",
    "ExtFig6_ap_per_class": "f97c7ff9f2ffda37d36dd8281e2701852d21684b699ee21ad631bfe09482f17f",
    "ExtFig7_split_influence": "8c0793e3c2abcfc4e05b45a4532ce64d9a77a26c5a9f9ec2ef3f8ad193d1934e",
}
FIGURE_SOURCES = {"Fig1": "Table 3 / ext01 / ext02", "Fig2": "Table 2 / Table 3 / ext11",
                  "ExtFig2_far_vs_recall": "threshold curves", "ExtFig3_rho_distribution": "ext07b",
                  "ExtFig5_expected_cost": "ext04", "ExtFig6_ap_per_class": "ext02",
                  "ExtFig7_split_influence": "ext10"}
for fig_name, source in FIGURE_SOURCES.items():
    path = FIGURE_DIR / f"{fig_name}.png"
    exists = path.is_file()
    _checks.append({"layer": "Figures", "item": f"{fig_name}.png", "computed": "written" if exists else "missing",
                    "printed": "", "tolerance": "", "status": "PASS" if exists else "FAIL",
                    "note": f"drawn from {source}"})
    if exists:
        same = hashlib.sha256(path.read_bytes()).hexdigest() == PUBLISHED_FIGURE_SHA256[fig_name]
        _checks.append({"layer": "Figure bytes (information)", "item": f"{fig_name}.png",
                        "computed": "identical" if same else "differs", "printed": "published file",
                        "tolerance": "SHA-256", "status": "PASS" if same else "INFO",
                        "note": "byte-identical to the published figure" if same else
                                "pixels differ from the published file (expected with another matplotlib version)"})

PAPER_CHECKS = pd.DataFrame(_checks)
INTEXT = PAPER_CHECKS[PAPER_CHECKS.layer == "In-text"][["item", "computed", "printed", "status", "note"]]
write_table(INTEXT.rename(columns={"item": "quantity", "note": "where"}), "intext_numbers")

In [ ]:
_artifact_rows = ARTIFACT_CHECKS.assign(layer="Original result tables").rename(columns={"check": "item", "detail": "note"})
VERIFICATION = pd.concat([_artifact_rows[["layer", "item", "status", "note"]],
                          PAPER_CHECKS[["layer", "item", "computed", "printed", "tolerance", "status", "note"]]],
                         ignore_index=True)
VERIFICATION.to_csv(OUTPUT_DIR / "verification_report.csv", index=False)

_summary = VERIFICATION.groupby("layer", sort=False).status.agg(
    checks="count", passed=lambda s: int((s == "PASS").sum()), failed=lambda s: int((s == "FAIL").sum()),
    information=lambda s: int((s == "INFO").sum()),
).reset_index()
show(_summary, "Verification summary")
_failures = VERIFICATION[VERIFICATION.status == "FAIL"]
if len(_failures):
    show(_failures, "Failed checks")
print(f"\n{len(VERIFICATION)} checks, {int((VERIFICATION.status == 'PASS').sum())} passed, {len(_failures)} failed. "
      f"Report: {OUTPUT_DIR / 'verification_report.csv'}")
print(f"Total run time: {(time.time() - NOTEBOOK_START) / 60:.1f} min")
if len(_failures):
    raise AssertionError(f"{len(_failures)} verification checks failed; see the table above.")
print("OVERALL: PASS")

## Appendix A. Retraining the YOLO detectors (optional)

Nothing in Sections 1 to 14 depends on this appendix. It documents how the stored predictions of the three YOLO detectors were produced, and lets a reader with a GPU regenerate them. It is switched off by `RUN_TRAINING = False`.

**What was locked.** All three YOLO detectors were trained through Ultralytics 8.4.124 (PyTorch 2.5.1, CUDA 12.4) on an RTX 4050 Laptop GPU, from the vendor's COCO-pretrained weights, with the settings below. They were fixed before the study began and are identical to the custom detector's where the two training loops allow it:

| Setting | Value |
|---|---|
| image size | 640 |
| epochs / early-stopping patience | 60 / 12 |
| batch size | 8 |
| optimiser | AdamW, lr0 = 0.001, final lr factor 0.001, weight decay 0.0005, cosine schedule, 3 warm-up epochs |
| mixed precision | on |
| augmentation | horizontal flip p = 0.5, scale 0.3, translation 0.1, HSV 0.015 / 0.5 / 0.4; no vertical flip, no rotation |
| **mosaic, mixup, copy-paste** | **off**: they synthesise composite images whose negative status is no longer human-verified |
| determinism | `deterministic=True`, seeds 42, 1337, 2024 |
| prediction | confidence ≥ 0.05, NMS IoU 0.60, default 300 detections per image |

**Data.** Ultralytics needs a YOLO-format copy of the partition. It is generated here from the same manifests the analysis uses, so every detector sees byte-identical splits; label files use this project's class order (0 = fire, 1 = smoke), which is the reverse of the order in D-Fire's own label files. Verified negatives get an empty label file, which is correct: those images were checked and contain nothing. The D-Fire images themselves are not part of this repository; download the release from https://github.com/gaiasd/DFireDataset and point `DFIRE_ROOT` at the folder that contains its `train/` and `test/` sub-folders.

**Predictions** are written in exactly the format the analysis reads (Section 2.4), so retrained runs can be dropped into `data/predictions/` and analysed with the cells above. A retrained run will not reproduce the stored predictions bit for bit: GPU kernels, library versions and data-loader timing all introduce small differences even with fixed seeds.

In [ ]:
RUN_TRAINING = False   # set to True to retrain (needs a CUDA GPU, the D-Fire images and `pip install -r requirements-training.txt`)

DFIRE_ROOT = Path(os.environ.get("DFIRE_ROOT", "DFireDataset")).expanduser().resolve()
TRAINING_DIR = OUTPUT_DIR / "training"
YOLO_DEVICE = os.environ.get("YOLO_DEVICE", "0")   # CUDA device index, or "cpu"
YOLO_WEIGHTS = {"YOLO11n": "yolo11n.pt", "YOLO11s": "yolo11s.pt", "YOLOv8s": "yolov8s.pt"}
YOLO_TRAIN_SETTINGS = {"epochs": 60, "imgsz": 640, "batch": 8, "deterministic": True, "plots": False, "val": True}
YOLO_HYPERPARAMETERS = {
    "mosaic": 0.0, "mixup": 0.0, "copy_paste": 0.0,
    "fliplr": 0.5, "flipud": 0.0, "degrees": 0.0, "scale": 0.3, "translate": 0.1,
    "hsv_h": 0.015, "hsv_s": 0.5, "hsv_v": 0.4,
    "optimizer": "AdamW", "lr0": 0.001, "lrf": 0.001, "weight_decay": 0.0005,
    "warmup_epochs": 3.0, "cos_lr": True, "patience": 12, "amp": True,
}
PREDICTION_CONFIDENCE = 0.05
PREDICTION_NMS_IOU = 0.60


def export_yolo_dataset(manifests: dict[str, pd.DataFrame], image_root: Path, out_dir: Path) -> Path:
    """Write a YOLO-format view of the given partitions and return the path of its data.yaml."""
    import shutil

    for split, frame in manifests.items():
        image_dir = out_dir / "images" / split
        label_dir = out_dir / "labels" / split
        image_dir.mkdir(parents=True, exist_ok=True)
        label_dir.mkdir(parents=True, exist_ok=True)
        for row in frame.itertuples(index=False):
            source = image_root / row.image_path
            if not source.is_file():
                raise FileNotFoundError(f"{source} is missing; check DFIRE_ROOT.")
            stem = row.sample_id.replace("/", "__")
            target = image_dir / f"{stem}{source.suffix}"
            if not target.exists():
                try:
                    os.link(source, target)        # hard link: no extra disk space
                except OSError:
                    shutil.copyfile(source, target)
            lines = []
            for cell in (row.fire_boxes, row.smoke_boxes):
                for b in _parse_boxes(cell):
                    lines.append(f"{CLASS_INDEX[b['cls']]} {b['cx']:.6f} {b['cy']:.6f} {b['w']:.6f} {b['h']:.6f}")
            # An empty file is meaningful here: the image is a verified negative.
            (label_dir / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
    data_yaml = out_dir / "data.yaml"
    names = "\n".join(f"  {i}: {c}" for i, c in enumerate(CLASSES))
    data_yaml.write_text(
        f"path: {out_dir.resolve().as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        f"nc: {len(CLASSES)}\nnames:\n{names}\n", encoding="utf-8")
    return data_yaml


def train_yolo(detector: str, seed: int, data_yaml: Path, run_name: str) -> Path:
    """Train one YOLO detector with the locked settings; returns the path of best.pt."""
    from ultralytics import YOLO

    model = YOLO(YOLO_WEIGHTS[detector])
    model.train(data=str(data_yaml), seed=seed, device=YOLO_DEVICE, project=str(TRAINING_DIR), name=run_name,
                exist_ok=True, verbose=True, **YOLO_TRAIN_SETTINGS, **YOLO_HYPERPARAMETERS)
    return TRAINING_DIR / run_name / "weights" / "best.pt"


def export_predictions(weights: Path, frame: pd.DataFrame, image_root: Path, split: str,
                       experiment: str, out_path: Path) -> None:
    """Predict every image of a partition and write the analysis's jsonl format."""
    from ultralytics import YOLO

    model = YOLO(str(weights))
    out_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    records = []
    for row in frame.itertuples(index=False):
        result = model.predict(str(image_root / row.image_path), conf=PREDICTION_CONFIDENCE,
                               iou=PREDICTION_NMS_IOU, imgsz=640, device=YOLO_DEVICE, verbose=False)[0]
        detections = []
        for box in result.boxes:
            idx = int(box.cls.item())
            detections.append({
                "bbox": [round(float(v), 2) for v in box.xyxy[0].tolist()],
                "score": round(float(box.conf.item()), 5),
                "class_idx": idx, "class": CLASSES[idx] if idx < len(CLASSES) else str(idx),
            })
        records.append({"sample_id": row.sample_id, "source": "dfire", "detections": detections})
    meta = {"_meta": {"split": split, "domain": "dfire", "experiment_id": experiment, "n_samples": len(records),
                      "conf_threshold_used": PREDICTION_CONFIDENCE,
                      "inference_seconds": round(time.time() - t0, 2)}}
    with out_path.open("w", encoding="utf-8") as handle:
        for record in [meta, *records]:
            handle.write(json.dumps(record) + "\n")


if RUN_TRAINING:
    _partitions = {"curated": MANIFESTS, "official": OFFICIAL_MANIFESTS}
    _yaml = {name: export_yolo_dataset(parts, DFIRE_ROOT, TRAINING_DIR / f"yolo_data_{name}")
             for name, parts in _partitions.items()}
    for run in RUNS.values():
        if run.detector not in YOLO_WEIGHTS:
            continue   # the custom detector is trained by its own pipeline (Appendix B)
        best = train_yolo(run.detector, run.seed, _yaml[run.partition], run.run_id)
        for split in ("val", "test"):
            export_predictions(best, _partitions[run.partition][split], DFIRE_ROOT, split, run.experiment,
                               TRAINING_DIR / "predictions" / run.stored_folder / f"{split}_dfire.jsonl")
    print(f"Predictions written to {TRAINING_DIR / 'predictions'}; set FIRE_PRED_DIR to that folder to analyse them.")
else:
    print("RUN_TRAINING is False: YOLO retraining skipped.")

## Appendix B. The custom ResNet18+FPN+FCOS detector (compact port)

The fourth detector was trained in the project's own pipeline (experiment E0b). Its full training code, with run registry, checkpointing, mixed-source batching and logging, is several thousand lines. The cells below port the parts that define the model and its learning problem, so that a reader can see exactly what was trained:

* **backbone**: ResNet18 (ImageNet-pretrained), features at strides 8, 16 and 32 (128, 256 and 512 channels);
* **neck**: a top-down feature pyramid with 1×1 lateral projections and a 3×3 Conv-BN-SiLU smoothing layer per level, 128 channels;
* **detection head**: FCOS style, anchor-free, one tower of two Conv-BN-SiLU layers shared by the classification, box and centerness branches, a learnable scale per level, box distances predicted as `exp(.) × stride`, focal-loss prior 0.01 on the class bias;
* **image-level presence head**: average-pooled stride-32 features, Linear(512, 256), SiLU, dropout 0.2, Linear(256, 3) for wildfire / fire / smoke presence. E0b trained it alongside the detector with weight 0.5, supervised only by presence labels derived from the boxes (the wildfire logit is masked out for D-Fire). It does not affect the detections;
* **target assignment**: FCOS rules with centre sampling (radius 1.5 strides), per-level size ranges [0, 64], [64, 128], [128, ∞) pixels, smallest box wins;
* **loss**: sigmoid focal loss (α = 0.25, γ = 2) normalised by positives, GIoU box loss weighted by centerness target (weight 2), centerness BCE (weight 1), computed per image;
* **post-processing**: score = √(class probability × centerness), top 1,000 candidates above 0.05, class-aware NMS at IoU 0.60, at most 300 detections, boxes mapped back from the 640 × 640 letterbox (pad value 114) to original pixels.

**Simplified or omitted, stated plainly.** (1) The original backbone came from `timm` (`resnet18`, timm 1.0.28 default ImageNet weights); this port uses torchvision's ResNet18, which has the same architecture and parameter count but different pretrained weights. (2) The training loop is reduced to a single step: the original also used gradient accumulation over 2 batches of 8, gradient clipping at norm 10, a per-step linear warm-up of 3 epochs followed by cosine decay to 1e-6, an exponential moving average of weights (decay 0.9998), bf16 autocast, checkpoint selection on validation mAP50-95 and early stopping after 12 epochs. (3) Data augmentation and letterboxing used OpenCV (area interpolation when shrinking); the letterbox here uses Pillow's bilinear resize and is provided for inference only. (4) Thermal input adapters, domain alignment and the self-supervised stage of the original pipeline are not used by E0b and are omitted.

The smoke test builds the model without downloading weights, runs one forward and backward pass on a dummy batch of two 640 × 640 images (one with boxes, one verified negative) on the CPU, decodes detections, and checks that the parameter count equals the 12.1703 M recorded for E0b in Table 2. It is switched off by `RUN_SMOKE_TEST = False`.

In [ ]:
RUN_SMOKE_TEST = False   # set to True to run one CPU forward/backward pass (needs torch and torchvision)

if RUN_SMOKE_TEST:
    import math as _math

    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision

    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD = (0.229, 0.224, 0.225)
    STRIDES = (8, 16, 32)
    SCALE_RANGES = ((0.0, 64.0), (64.0, 128.0), (128.0, 1e8))
    INF = 1e8

    class ConvNormAct(nn.Sequential):
        def __init__(self, cin: int, cout: int, k: int = 3) -> None:
            super().__init__(nn.Conv2d(cin, cout, k, 1, padding=k // 2, bias=False), nn.BatchNorm2d(cout),
                             nn.SiLU(inplace=True))

    class ResNet18Features(nn.Module):
        """ResNet18 trunk returning stride-8, -16 and -32 feature maps."""

        def __init__(self, pretrained: bool) -> None:
            super().__init__()
            weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            net = torchvision.models.resnet18(weights=weights)
            self.stem = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool, net.layer1)
            self.layer2, self.layer3, self.layer4 = net.layer2, net.layer3, net.layer4
            self.channels = [128, 256, 512]

        def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
            c3 = self.layer2(self.stem(x))
            c4 = self.layer3(c3)
            return [c3, c4, self.layer4(c4)]

    class FPN(nn.Module):
        def __init__(self, in_channels: list[int], out_channels: int = 128) -> None:
            super().__init__()
            self.lateral = nn.ModuleList(nn.Conv2d(c, out_channels, 1) for c in in_channels)
            self.smooth = nn.ModuleList(ConvNormAct(out_channels, out_channels) for _ in in_channels)
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_uniform_(m.weight, a=1)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)

        def forward(self, feats: list[torch.Tensor]) -> list[torch.Tensor]:
            lat = [conv(f) for conv, f in zip(self.lateral, feats)]
            for i in range(len(lat) - 1, 0, -1):
                lat[i - 1] = lat[i - 1] + F.interpolate(lat[i], size=lat[i - 1].shape[-2:], mode="nearest")
            return [s(x) for s, x in zip(self.smooth, lat)]

    class Scale(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.scale = nn.Parameter(torch.tensor(1.0))

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return x * self.scale

    class FCOSHead(nn.Module):
        def __init__(self, channels: int = 128, num_classes: int = 2, prior_prob: float = 0.01) -> None:
            super().__init__()
            self.tower = nn.Sequential(ConvNormAct(channels, channels), ConvNormAct(channels, channels))
            self.cls_pred = nn.Conv2d(channels, num_classes, 3, padding=1)
            self.reg_pred = nn.Conv2d(channels, 4, 3, padding=1)
            self.ctr_pred = nn.Conv2d(channels, 1, 3, padding=1)
            self.scales = nn.ModuleList(Scale() for _ in STRIDES)
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.normal_(m.weight, std=0.01)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
            nn.init.constant_(self.cls_pred.bias, -_math.log((1 - prior_prob) / prior_prob))

        def forward(self, feats: list[torch.Tensor]) -> dict[str, list[torch.Tensor]]:
            out: dict[str, list[torch.Tensor]] = {"cls": [], "ltrb": [], "ctr": [], "points": []}
            for i, f in enumerate(feats):
                t = self.tower(f)
                out["cls"].append(self.cls_pred(t))
                # exp() keeps distances positive; the stride puts the initial box inside the level's size band.
                out["ltrb"].append(torch.exp(self.scales[i](self.reg_pred(t))) * STRIDES[i])
                out["ctr"].append(self.ctr_pred(t))
                h, w = f.shape[-2:]
                ys = torch.arange(h, dtype=torch.float32) * STRIDES[i] + STRIDES[i] / 2
                xs = torch.arange(w, dtype=torch.float32) * STRIDES[i] + STRIDES[i] / 2
                yy, xx = torch.meshgrid(ys, xs, indexing="ij")
                out["points"].append(torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1))
            return out

    class PresenceHead(nn.Module):
        """Image-level wildfire / fire / smoke presence logits from pooled stride-32 features."""

        def __init__(self, in_channels: int = 512, hidden: int = 256, outputs: int = 3) -> None:
            super().__init__()
            self.mlp = nn.Sequential(nn.Linear(in_channels, hidden), nn.SiLU(inplace=True), nn.Dropout(0.2),
                                     nn.Linear(hidden, outputs))
            for m in self.mlp.modules():
                if isinstance(m, nn.Linear):
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    nn.init.zeros_(m.bias)

        def forward(self, c5: torch.Tensor) -> torch.Tensor:
            return self.mlp(c5.mean(dim=(2, 3)))

    class ResNetFPNFCOS(nn.Module):
        def __init__(self, pretrained: bool = True) -> None:
            super().__init__()
            self.encoder = ResNet18Features(pretrained)
            self.neck = FPN(self.encoder.channels, 128)
            self.det_head = FCOSHead(128, len(CLASSES))
            self.cls_head = PresenceHead(self.encoder.channels[-1])

        def forward(self, images: torch.Tensor) -> tuple[dict[str, list[torch.Tensor]], torch.Tensor]:
            feats = self.encoder(images)
            return self.det_head(self.neck(feats)), self.cls_head(feats[-1])

        def param_groups(self, head_lr: float = 1e-3, backbone_lr: float = 1e-4, weight_decay: float = 5e-4) -> list[dict]:
            """Backbone and head learning rates; no weight decay on norm layers and biases."""
            groups: dict[tuple[str, bool], list[nn.Parameter]] = {}
            for name, p in self.named_parameters():
                where = "backbone" if name.startswith("encoder.") else "head"
                groups.setdefault((where, p.ndim <= 1 or name.endswith(".bias")), []).append(p)
            return [{"params": ps, "lr": backbone_lr if where == "backbone" else head_lr,
                     "weight_decay": 0.0 if no_decay else weight_decay}
                    for (where, no_decay), ps in groups.items()]

    def flatten_levels(tensors: list[torch.Tensor]) -> torch.Tensor:
        """(N, C, H, W) per level -> (N, sum(H*W), C)."""
        return torch.cat([t.permute(0, 2, 3, 1).reshape(t.shape[0], -1, t.shape[1]) for t in tensors], dim=1)

    def centerness_target(ltrb: torch.Tensor) -> torch.Tensor:
        lr, tb = ltrb[:, [0, 2]], ltrb[:, [1, 3]]
        return torch.sqrt((lr.min(1).values.clamp(min=0) / lr.max(1).values.clamp(min=1e-6))
                          * (tb.min(1).values.clamp(min=0) / tb.max(1).values.clamp(min=1e-6)))

    def assign_targets(points_per_level: list[torch.Tensor], gt: torch.Tensor) -> tuple[torch.Tensor, ...]:
        """FCOS assignment for one image. gt: (G, 5) rows of [x1, y1, x2, y2, class]."""
        points = torch.cat(points_per_level)
        n = points.shape[0]
        ranges = torch.cat([p.new_tensor(SCALE_RANGES[i]).expand(p.shape[0], 2) for i, p in enumerate(points_per_level)])
        strides = torch.cat([p.new_full((p.shape[0],), STRIDES[i]) for i, p in enumerate(points_per_level)])
        labels = torch.full((n,), len(CLASSES), dtype=torch.long)
        if gt.numel() == 0:
            return labels, torch.zeros(n, 4), torch.zeros(n), torch.zeros(n, dtype=torch.bool)
        x, y = points[:, 0:1], points[:, 1:2]
        x1, y1, x2, y2 = (gt[:, i].unsqueeze(0) for i in range(4))
        ltrb = torch.stack([x - x1, y - y1, x2 - x, y2 - y], dim=2)
        inside = ltrb.min(dim=2).values > 0
        cx, cy, radius = (x1 + x2) / 2, (y1 + y2) / 2, strides.unsqueeze(1) * 1.5
        inside &= (x > torch.maximum(cx - radius, x1)) & (x < torch.minimum(cx + radius, x2)) \
            & (y > torch.maximum(cy - radius, y1)) & (y < torch.minimum(cy + radius, y2))
        side = ltrb.max(dim=2).values
        in_range = (side >= ranges[:, 0:1]) & (side <= ranges[:, 1:2])
        areas = ((x2 - x1) * (y2 - y1)).expand(n, gt.shape[0]).clone()
        areas[~(inside & in_range)] = INF
        min_area, idx = areas.min(dim=1)
        pos = min_area < INF
        labels[pos] = gt[idx[pos], 4].long()
        targets = torch.zeros(n, 4)
        targets[pos] = ltrb[torch.arange(n), idx][pos]
        ctr = torch.zeros(n)
        if pos.any():
            ctr[pos] = centerness_target(targets[pos])
        return labels, targets, ctr, pos

    def focal_loss(logits: torch.Tensor, targets: torch.Tensor, alpha: float = 0.25, gamma: float = 2.0) -> torch.Tensor:
        p = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = p * targets + (1 - p) * (1 - targets)
        return (alpha * targets + (1 - alpha) * (1 - targets)) * ce * (1 - p_t) ** gamma

    def giou_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
        """GIoU loss on (left, top, right, bottom) distances from the same location."""
        pl, pt, pr, pb = pred.unbind(-1)
        tl, tt, tr, tb = target.unbind(-1)
        area_p = (pl + pr).clamp(min=0) * (pt + pb).clamp(min=0)
        area_t = (tl + tr).clamp(min=0) * (tt + tb).clamp(min=0)
        inter = (torch.min(pl, tl) + torch.min(pr, tr)).clamp(min=0) * (torch.min(pt, tt) + torch.min(pb, tb)).clamp(min=0)
        union = area_p + area_t - inter + eps
        enclose = (torch.max(pl, tl) + torch.max(pr, tr)).clamp(min=0) * (torch.max(pt, tt) + torch.max(pb, tb)).clamp(min=0) + eps
        return 1.0 - (inter / union - (enclose - union) / enclose)

    def detection_loss(out: dict[str, list[torch.Tensor]], gt_boxes: list[torch.Tensor]) -> torch.Tensor:
        """Per-image FCOS loss (focal + 2 x centerness-weighted GIoU + centerness BCE), averaged over images."""
        cls, box, ctr = flatten_levels(out["cls"]), flatten_levels(out["ltrb"]), flatten_levels(out["ctr"])
        losses = []
        for i, gt in enumerate(gt_boxes):
            labels, targets, ctr_t, pos = assign_targets(out["points"], gt)
            onehot = torch.zeros_like(cls[i])
            onehot[pos, labels[pos]] = 1.0
            l_cls = focal_loss(cls[i], onehot).sum() / max(1, int(pos.sum()))
            if pos.any():
                l_box = (giou_loss(box[i][pos], targets[pos]) * ctr_t[pos]).sum() / ctr_t[pos].sum().clamp(min=1e-6)
                l_ctr = F.binary_cross_entropy_with_logits(ctr[i][pos].squeeze(-1), ctr_t[pos])
            else:
                l_box = l_ctr = cls.new_zeros(())
            losses.append(1.0 * l_cls + 2.0 * l_box + 1.0 * l_ctr)
        return torch.stack(losses).mean()

    def presence_loss(logits: torch.Tensor, targets: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        """BCE over the presence logits that carry a label; unknown logits contribute nothing."""
        per_logit = F.binary_cross_entropy_with_logits(logits, targets, reduction="none") * masks
        return (per_logit.sum(1) / masks.sum(1).clamp(min=1.0)).mean()

    @torch.no_grad()
    def decode(out: dict[str, list[torch.Tensor]], letterbox_meta: list[dict[str, float]], conf: float = 0.05,
               nms_iou: float = 0.60, max_det: int = 300, topk: int = 1000) -> list[list[dict[str, Any]]]:
        """Head outputs -> detections in original-image pixels, in the stored-prediction format."""
        cls, box, ctr = flatten_levels(out["cls"]), flatten_levels(out["ltrb"]), flatten_levels(out["ctr"])
        points = torch.cat(out["points"])
        scores_all = (cls.sigmoid() * ctr.sigmoid()).sqrt()
        results = []
        for i in range(cls.shape[0]):
            flat = scores_all[i].reshape(-1)
            k = min(topk, int((flat > conf).sum()))
            if k == 0:
                results.append([])
                continue
            top_scores, top_idx = flat.topk(k)
            p_idx, c_idx = top_idx // len(CLASSES), top_idx % len(CLASSES)
            pts, d = points[p_idx], box[i][p_idx]
            boxes = torch.stack([pts[:, 0] - d[:, 0], pts[:, 1] - d[:, 1], pts[:, 0] + d[:, 2], pts[:, 1] + d[:, 3]], 1)
            keep = torchvision.ops.batched_nms(boxes, top_scores, c_idx, nms_iou)[:max_det]
            m = letterbox_meta[i]
            b = boxes[keep].clone()
            b[:, [0, 2]] = ((b[:, [0, 2]] - m["pad_x"]) / m["scale"]).clamp(0, m["orig_w"])
            b[:, [1, 3]] = ((b[:, [1, 3]] - m["pad_y"]) / m["scale"]).clamp(0, m["orig_h"])
            results.append([{"bbox": [round(float(v), 2) for v in bb], "score": round(float(s), 5),
                             "class_idx": int(c), "class": CLASSES[int(c)]}
                            for bb, s, c in zip(b, top_scores[keep], c_idx[keep])])
        return results

    def letterbox(image: "Image.Image", size: int = 640, pad_value: int = 114) -> tuple[torch.Tensor, dict[str, float]]:
        """Aspect-preserving resize into a padded square, normalised with ImageNet statistics (inference only)."""
        w, h = image.size
        scale = min(size / w, size / h)
        nw, nh = max(1, round(w * scale)), max(1, round(h * scale))
        canvas = np.full((size, size, 3), pad_value, dtype=np.uint8)
        pad_x, pad_y = (size - nw) / 2, (size - nh) / 2
        canvas[round(pad_y):round(pad_y) + nh, round(pad_x):round(pad_x) + nw] = np.asarray(image.convert("RGB").resize((nw, nh)))
        x = torch.from_numpy(canvas).permute(2, 0, 1).float() / 255.0
        x = (x - torch.tensor(IMAGENET_MEAN)[:, None, None]) / torch.tensor(IMAGENET_STD)[:, None, None]
        return x, {"scale": scale, "pad_x": pad_x, "pad_y": pad_y, "orig_w": w, "orig_h": h}

    # --- Smoke test: one forward and backward pass on the CPU -----------------------------------
    torch.manual_seed(RANDOM_SEED)
    model = ResNetFPNFCOS(pretrained=False)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"parameters: {n_params:,} ({n_params / 1e6:.4f} M; recorded for E0b: 12.1703 M)")
    assert round(n_params / 1e6, 4) == 12.1703, "parameter count differs from the trained detector"

    optimizer = torch.optim.AdamW(model.param_groups(), betas=(0.9, 0.999))
    images = torch.randn(2, 3, 640, 640)
    gt_boxes = [torch.tensor([[100.0, 120.0, 180.0, 220.0, 0.0], [300.0, 50.0, 600.0, 400.0, 1.0]]),
                torch.zeros((0, 5))]                                   # second image: a verified negative
    presence_targets = torch.tensor([[0.0, 1.0, 1.0], [0.0, 0.0, 0.0]])
    presence_masks = torch.tensor([[0.0, 1.0, 1.0], [0.0, 1.0, 1.0]])  # wildfire_present unknown on D-Fire

    model.train()
    det_out, presence_logits = model(images)
    loss = detection_loss(det_out, gt_boxes) + 0.5 * presence_loss(presence_logits, presence_targets, presence_masks)
    optimizer.zero_grad()
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
    optimizer.step()
    assert torch.isfinite(loss) and torch.isfinite(grad_norm)
    print(f"loss {loss.item():.4f}, gradient norm before clipping {grad_norm.item():.3f}")

    model.eval()
    from PIL import Image

    _dummy = Image.fromarray((np.random.default_rng(RANDOM_SEED).random((450, 800, 3)) * 255).astype(np.uint8))
    _x, _meta = letterbox(_dummy)
    _dets = decode(model(_x[None])[0], [_meta])
    # Before training every location scores about sqrt(0.01 x 0.5) = 0.07, above the 0.05 floor,
    # so the 300-detection cap is reached; the count only shows that decoding and NMS run.
    print(f"decoded {len(_dets[0])} detections on a random 800 x 450 image")
    print("Smoke test passed.")
else:
    print("RUN_SMOKE_TEST is False: model smoke test skipped.")